# Moons, Stars, and Clever Hans: What Did the Model Actually Learn?

> Accuracy says both models learned the task. Moving the same moon shows one of them learned the wrong thing.


## 1. Title and hook

This notebook is a small Clever Hans laboratory. Ordinary validation asks whether the model is right. Behavioural XAI asks whether it is right for the right reason.

The framing follows the warning from Lapuschkin et al., [“Unmasking Clever Hans predictors and assessing what machines really learn”](https://www.nature.com/articles/s41467-019-08987-4): high predictive accuracy can hide the strategy a model has actually learned. It also treats saliency cautiously, in the spirit of Adebayo et al., [“Sanity Checks for Saliency Maps”](https://papers.nips.cc/paper/2018/hash/294a8ed24b1ad22ec2e7efea049b8737-Abstract.html): a plausible visual explanation is not enough by itself.

We will start the way a real audit often starts: the task looks simple, the models look strong, and nothing seems wrong. Then we move the same object and let the model betray its rule.


## 2. The apparent task

The visible task is innocent: classify the shape as `moon` or `star`. At this point, do not think about shortcuts. Imagine you only see small generated shape images and a balanced-looking audit sample.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig00_apparent_shape_task.png" style="width:100%; border:1px solid #ddd">


### The investigation

1. **Apparent success:** both models look excellent on ordinary validation.
2. **First crack:** moving the same shape exposes a wrong rule.
3. **Hidden exam leak:** only after the failure do we inspect the data-generating process.
4. **Interrogation:** response maps, movement paths, morphs, and saliency ask what controls the score.
5. **Verdict:** multiple probes show that the MLP learned location, while the CNN is more shape-stable.

Saliency is supporting evidence only. The central evidence is behavioural: same shape, moved position, changed model score, then re-tested after intervention.


In [ ]:
# ruff: noqa
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFilter, ImageFont

SEED = 1701
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)

plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "figure.dpi": 140,
        "savefig.dpi": 180,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def find_project_root() -> Path:
    """Return the repository root without assuming the notebook working directory."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the XAI project root.")


def find_notebook_path() -> Path:
    """Locate this notebook from either repo root or notebooks/shortcut_lab."""
    candidates = [
        Path.cwd() / "00_moons_stars_clever_hans.ipynb",
        Path.cwd() / "notebooks/shortcut_lab/00_moons_stars_clever_hans.ipynb",
        find_project_root() / "notebooks/shortcut_lab/00_moons_stars_clever_hans.ipynb",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate 00_moons_stars_clever_hans.ipynb")


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = find_notebook_path()
FIGURE_DIR = PROJECT_ROOT / "outputs/notebook_figures/00_moons_stars_clever_hans"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DEMO = "00 Moons/Stars Clever-Hans position shortcut"
DATA_MODE = "generated_controlled_demo"
EXTERNAL_DATA_REQUIRED = False
MANIFEST_PATH = "not applicable"
MANIFEST_EXISTS = False
DATASET_SOURCE = "Generated inside this notebook"
LICENCE_NOTE = "Repo-authored controlled generated data; no external dataset licence."
MISSING_FILES: list[str] = []
TRUE_TASK = "shape: moon versus star"
SHORTCUT = "absolute object position"
TRAINING_BIAS = "moons lower-left; stars upper-right"
BASELINE_MODEL = "Pixel MLP"
ROBUST_MODEL = "translation-aware CNN"
XAI_PROBES = "movement counterfactuals, position-response maps, shape morphs, decision surfaces, saliency, attribution mass, evidence removal"

status_lines = [
    "DEMO: 00 Moons, Stars, and Clever Hans",
    f"DATA_MODE: {DATA_MODE}",
    f"EXTERNAL_DATA_REQUIRED: {str(EXTERNAL_DATA_REQUIRED).lower()}",
    "TRUE TASK: shape classification",
    "AUDIT QUESTION: did the model learn the shape, or a hidden rule?",
    "BASELINE MODEL: pixel MLP",
    "ROBUST MODEL: translation-aware CNN",
    "XAI PROBES: movement counterfactuals, confidence paths, response maps, morphs, saliency caution, re-test",
    f"MANIFEST_PATH: {MANIFEST_PATH}",
    f"MANIFEST_EXISTS: {MANIFEST_EXISTS}",
    f"PROJECT_ROOT: {PROJECT_ROOT}",
    f"DATASET_SOURCE: {DATASET_SOURCE}",
    f"LICENCE_NOTE: {LICENCE_NOTE}",
    f"MISSING_FILES: {MISSING_FILES}",
    f"SEED: {SEED}",
]
display(Markdown("```text\n" + "\n".join(status_lines) + "\n```"))


## 2.1 Generate the controlled laboratory

The code below generates all images inside the notebook. There is no external dataset. The visible labels are only `moon` and `star`; the shortcut is deliberately withheld from the reader until the behavioural failure appears.

The background is neutral and not class-correlated. Shape size, rotation, brightness, stroke, crescent thickness, texture, and position all vary.


In [ ]:
IMAGE_SIZE = 128
MODEL_SIZE = 40
RENDER_SCALE = 3
CLASS_NAMES = {0: "moon", 1: "star"}
CLASS_COLOURS = {0: "#4C78A8", 1: "#F2A541"}
MODEL_COLOURS = {"MLP": "#C44E52", "CNN": "#2E8B57"}
LOWER_LEFT_REGION = (24, 48, 82, 108)  # x_min, x_max, y_min, y_max
UPPER_RIGHT_REGION = (82, 108, 18, 44)
FREE_REGION = (24, 108, 18, 108)
CANONICAL_CENTRES = {"lower_left": (36, 96), "upper_right": (96, 32)}
COMMON_GROUPS = {"moon_lower_left", "star_upper_right"}
CROSSED_GROUPS = {"moon_upper_right", "star_lower_left"}


@dataclass(frozen=True)
class ShapeSpec:
    label: int
    centre: tuple[int, int]
    seed: int
    size: float
    rotation: float
    brightness: int
    stroke: int
    crescent: float


@dataclass(frozen=True)
class PositionSample:
    image: Image.Image
    label: int
    position: str
    centre: tuple[int, int]
    group: str
    split: str
    object_seed: int
    spec: ShapeSpec


def rng_for(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


def position_name(centre: tuple[int, int]) -> str:
    x, y = centre
    if x < IMAGE_SIZE // 2 and y >= IMAGE_SIZE // 2:
        return "lower_left"
    if x >= IMAGE_SIZE // 2 and y < IMAGE_SIZE // 2:
        return "upper_right"
    return "uniform"


def group_name(label: int, position: str) -> str:
    return f"{CLASS_NAMES[label]}_{position}"


def choose_centre(region: tuple[int, int, int, int], rng: np.random.Generator) -> tuple[int, int]:
    return (
        int(rng.integers(region[0], region[1] + 1)),
        int(rng.integers(region[2], region[3] + 1)),
    )


def make_shape_spec(
    label: int,
    region: tuple[int, int, int, int],
    seed: int,
    fixed_centre: tuple[int, int] | None = None,
) -> ShapeSpec:
    local_rng = rng_for(seed)
    centre = fixed_centre if fixed_centre is not None else choose_centre(region, local_rng)
    return ShapeSpec(
        label=label,
        centre=centre,
        seed=seed,
        size=float(local_rng.uniform(20.0, 24.0)),
        rotation=float(local_rng.uniform(-16.0, 16.0)),
        brightness=int(local_rng.integers(228, 256)),
        stroke=int(local_rng.integers(1, 2)),
        crescent=float(local_rng.uniform(0.60, 0.70)),
    )


def move_spec(spec: ShapeSpec, centre: tuple[int, int]) -> ShapeSpec:
    return ShapeSpec(
        label=spec.label,
        centre=centre,
        seed=spec.seed,
        size=spec.size,
        rotation=spec.rotation,
        brightness=spec.brightness,
        stroke=spec.stroke,
        crescent=spec.crescent,
    )


def star_points(cx: float, cy: float, outer: float, inner: float, rotation: float) -> list[tuple[float, float]]:
    points = []
    for index in range(10):
        radius = outer if index % 2 == 0 else inner
        angle = math.radians(rotation - 90.0 + index * 36.0)
        points.append((cx + math.cos(angle) * radius, cy + math.sin(angle) * radius))
    return points


def object_mask(spec: ShapeSpec) -> Image.Image:
    canvas_size = IMAGE_SIZE * RENDER_SCALE
    mask = Image.new("L", (canvas_size, canvas_size), 0)
    draw = ImageDraw.Draw(mask)
    cx = spec.centre[0] * RENDER_SCALE
    cy = spec.centre[1] * RENDER_SCALE
    size = spec.size * RENDER_SCALE
    if spec.label == 1:
        draw.polygon(star_points(cx, cy, size, size * 0.45, spec.rotation), fill=255)
    else:
        draw.ellipse((cx - size, cy - size, cx + size, cy + size), fill=255)
        offset = size * spec.crescent
        draw.ellipse((cx - size + offset, cy - size * 1.06, cx + size + offset, cy + size * 1.06), fill=0)
    return mask.filter(ImageFilter.GaussianBlur(0.6 * RENDER_SCALE)).resize(
        (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS
    )


def render_shape(spec: ShapeSpec) -> Image.Image:
    local_rng = rng_for(spec.seed + 99)
    yy, xx = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE]
    texture = (
        38.0
        + 1.8 * np.sin(xx / 11.0)
        + 1.3 * np.cos(yy / 13.0)
        + local_rng.normal(0.0, 1.3, (IMAGE_SIZE, IMAGE_SIZE))
    )
    background = Image.fromarray(np.clip(texture, 0, 255).astype(np.uint8), "L")
    object_layer = Image.fromarray(np.full((IMAGE_SIZE, IMAGE_SIZE), spec.brightness, dtype=np.uint8), "L")
    return Image.composite(object_layer, background, object_mask(spec))


def make_sample(
    label: int,
    region: tuple[int, int, int, int],
    split: str,
    seed: int,
    fixed_centre: tuple[int, int] | None = None,
) -> PositionSample:
    spec = make_shape_spec(label, region, seed, fixed_centre=fixed_centre)
    position = position_name(spec.centre)
    return PositionSample(
        image=render_shape(spec),
        label=label,
        position=position,
        centre=spec.centre,
        group=group_name(label, position),
        split=split,
        object_seed=seed,
        spec=spec,
    )


def make_dataset(
    split: str,
    common_per_class: int,
    crossed_per_class: int,
    seed_offset: int,
    free_positions: bool = False,
) -> list[PositionSample]:
    samples: list[PositionSample] = []
    counter = seed_offset
    for label in (0, 1):
        common_region = LOWER_LEFT_REGION if label == 0 else UPPER_RIGHT_REGION
        crossed_region = UPPER_RIGHT_REGION if label == 0 else LOWER_LEFT_REGION
        for _ in range(common_per_class):
            region = FREE_REGION if free_positions else common_region
            samples.append(make_sample(label, region, split, SEED + counter))
            counter += 1
        for _ in range(crossed_per_class):
            region = FREE_REGION if free_positions else crossed_region
            samples.append(make_sample(label, region, split, SEED + counter))
            counter += 1
    random.Random(SEED + seed_offset).shuffle(samples)
    return samples


biased_train_samples = make_dataset("biased train", 600, 0, seed_offset=0)
iid_validation_samples = make_dataset("IID validation", 200, 0, seed_offset=10_000)
ood_audit_samples = make_dataset("balanced OOD audit", 60, 60, seed_offset=20_000)
position_augmented_train_samples = make_dataset(
    "position-augmented train", 300, 300, seed_offset=30_000, free_positions=True
)
position_augmented_validation_samples = make_dataset(
    "position-augmented validation", 120, 120, seed_offset=40_000, free_positions=True
)
uniform_audit_samples = make_dataset("uniform-position audit", 120, 0, seed_offset=50_000, free_positions=True)

print(f"Biased training samples: {len(biased_train_samples)}")
print(f"IID validation samples: {len(iid_validation_samples)}")
print(f"Balanced OOD audit samples: {len(ood_audit_samples)}")
print(f"position-augmented train samples: {len(position_augmented_train_samples)}")
print(f"Uniform-position audit samples: {len(uniform_audit_samples)}")


In [ ]:
def save_and_show(fig: plt.Figure, filename: str) -> None:
    fig.savefig(FIGURE_DIR / filename, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def image_array(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("L"), dtype=np.float32) / 255.0


def representative_sample(label: int, position: str) -> PositionSample:
    centre = CANONICAL_CENTRES[position]
    region = LOWER_LEFT_REGION if position == "lower_left" else UPPER_RIGHT_REGION
    return make_sample(label, region, "representative", SEED + 70_000 + label * 100, fixed_centre=centre)


def sample_at_centre(label: int, centre: tuple[int, int], seed: int, split: str = "presentation") -> PositionSample:
    region = (centre[0], centre[0], centre[1], centre[1])
    return make_sample(label, region, split, seed, fixed_centre=centre)


# Innocent-looking task figure. This deliberately avoids showing the training-position distribution.
apparent_examples = [
    sample_at_centre(0, (34, 36), SEED + 71_001),
    sample_at_centre(0, (64, 72), SEED + 71_002),
    sample_at_centre(0, (92, 42), SEED + 71_003),
    sample_at_centre(1, (38, 88), SEED + 71_101),
    sample_at_centre(1, (68, 34), SEED + 71_102),
    sample_at_centre(1, (96, 86), SEED + 71_103),
]
fig = plt.figure(figsize=(11.0, 6.2), constrained_layout=True)
grid = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 1.15])
fig.suptitle("The task humans think we gave the model", fontsize=19, fontweight="bold")
fig.text(0.5, 0.925, "Classify shape: moon or star.", ha="center", fontsize=13.5, color="#555555")
for idx, sample in enumerate(apparent_examples[:3]):
    axis = fig.add_subplot(grid[0, idx])
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_title("moon example", color=CLASS_COLOURS[0], fontweight="bold")
    axis.set_xticks([])
    axis.set_yticks([])
for idx, sample in enumerate(apparent_examples[3:]):
    axis = fig.add_subplot(grid[1, idx])
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_title("star example", color=CLASS_COLOURS[1], fontweight="bold")
    axis.set_xticks([])
    axis.set_yticks([])
audit_axis = fig.add_subplot(grid[:, 3])
audit_axis.axis("off")
audit_axis.add_patch(plt.Rectangle((0.04, 0.08), 0.92, 0.84, facecolor="#F8F8F8", edgecolor="#D0D0D0", linewidth=1.4, transform=audit_axis.transAxes))
audit_axis.text(0.11, 0.82, "Balanced-looking audit", fontsize=13.5, fontweight="bold", transform=audit_axis.transAxes)
audit_axis.text(0.11, 0.64, "moons and stars\nappear visually clear", fontsize=12, transform=audit_axis.transAxes)
audit_axis.text(0.11, 0.42, "The apparent question:\ncan the model recognise\nthe shape?", fontsize=12, transform=audit_axis.transAxes)
audit_axis.text(0.11, 0.18, "No external data.\nGenerated controlled demo.", fontsize=10.5, color="#666666", transform=audit_axis.transAxes)
fig.text(0.5, -0.02, "At this point the task looks like ordinary shape recognition.", ha="center", fontsize=12, fontweight="bold")
save_and_show(fig, "fig00_apparent_shape_task.png")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.8), constrained_layout=True)
fig.suptitle("Two learned models, two different incentives", fontsize=18, fontweight="bold")
model_cards = [
    (
        axes[0],
        "Pixel MLP",
        ["input image", "flatten pixels", "dense layers", "moon / star output"],
        "Can use absolute pixel address.",
        MODEL_COLOURS["MLP"],
    ),
    (
        axes[1],
        "CNN + augmentation",
        ["input image", "local filters", "global average pooling", "moon / star output"],
        "Encouraged to learn local shape features.",
        MODEL_COLOURS["CNN"],
    ),
]
for axis, title, steps, note, colour in model_cards:
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.02, 0.06), 0.96, 0.86, facecolor="#FAFAFA", edgecolor=colour, linewidth=2.4, transform=axis.transAxes))
    axis.text(0.08, 0.82, title, fontsize=16, fontweight="bold", color=colour, transform=axis.transAxes)
    x_positions = np.linspace(0.14, 0.82, len(steps))
    for index, (xpos, step) in enumerate(zip(x_positions, steps, strict=True)):
        axis.add_patch(plt.Circle((xpos, 0.52), 0.075, facecolor="white", edgecolor=colour, linewidth=2.0, transform=axis.transAxes))
        axis.text(xpos, 0.52, str(index + 1), ha="center", va="center", fontsize=12, fontweight="bold", color=colour, transform=axis.transAxes)
        axis.text(xpos, 0.34, step, ha="center", va="center", fontsize=10.5, transform=axis.transAxes, wrap=True)
        if index < len(steps) - 1:
            axis.annotate("", xy=(x_positions[index + 1] - 0.09, 0.52), xytext=(xpos + 0.09, 0.52), xycoords=axis.transAxes, arrowprops={"arrowstyle": "->", "lw": 1.7, "color": "#555555"})
    axis.text(0.08, 0.16, note, fontsize=12.2, fontweight="bold", color=colour, transform=axis.transAxes)
fig.text(0.5, -0.02, "Both are learned neural networks. The difference is the incentive each architecture receives.", ha="center", fontsize=12, fontweight="bold")
save_and_show(fig, "fig00_model_cards.png")


## 3. Model structure

We compare two learned models. The `PixelMLP` flattens the image and can assign different meanings to different pixel addresses. The `GapCNN` uses shared convolutional filters, global average pooling, and position augmentation, so it is encouraged to recognise local shape features across the canvas.

The CNN is not magically translation invariant. It is more translation-aware in this controlled setting.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig00_model_cards.png" style="width:100%; border:1px solid #ddd">


In [ ]:

TENSOR_CACHE: dict[tuple[int, ...], tuple[torch.Tensor, torch.Tensor]] = {}
ACT1_TARGET_MARGIN = 6.0
ACT1_CONFIDENCE_MEAN_THRESHOLD = 0.98
ACT1_CONFIDENCE_MIN_THRESHOLD = 0.90


def samples_to_tensors(samples: list[PositionSample]) -> tuple[torch.Tensor, torch.Tensor]:
    cache_key = tuple(id(sample) for sample in samples)
    if cache_key in TENSOR_CACHE:
        return TENSOR_CACHE[cache_key]
    arrays = [
        np.asarray(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
        for sample in samples
    ]
    x = torch.tensor(np.stack(arrays)[:, None, :, :], dtype=torch.float32)
    y = torch.tensor([sample.label for sample in samples], dtype=torch.long)
    TENSOR_CACHE[cache_key] = (x, y)
    return x, y


class PixelMLP(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(MODEL_SIZE * MODEL_SIZE, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GapCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 24, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(24, 48, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(48, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d(1),
        )
        self.head = torch.nn.Linear(64, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x).flatten(1))


def binary_correct_class_probabilities(probabilities: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    row_ids = torch.arange(labels.shape[0], device=labels.device)
    return probabilities[row_ids, labels]


def act1_margin_loss(logits: torch.Tensor, y: torch.Tensor, target_margin: float = ACT1_TARGET_MARGIN) -> torch.Tensor:
    correct = logits.gather(1, y[:, None]).squeeze(1)
    wrong = logits.gather(1, (1 - y)[:, None]).squeeze(1)
    return torch.relu(target_margin - (correct - wrong)).mean()


def evaluate_model(model: torch.nn.Module, samples: list[PositionSample]) -> dict[str, Any]:
    x, y = samples_to_tensors(samples)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(logits, y).item()
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)
        correct_probs = binary_correct_class_probabilities(probabilities, y)
    return {
        "loss": loss,
        "accuracy": float((predictions == y).float().mean().item()),
        "mean_correct_prob": float(correct_probs.mean().item()),
        "min_correct_prob": float(correct_probs.min().item()),
        "star_score": probabilities[:, 1].cpu().numpy(),
        "correct_prob": correct_probs.cpu().numpy(),
        "predictions": predictions.cpu().numpy(),
        "labels": y.cpu().numpy(),
    }


def act1_confidence_ready(train_eval: dict[str, Any], iid_eval: dict[str, Any]) -> bool:
    return (
        train_eval["accuracy"] >= 0.995
        and iid_eval["accuracy"] >= 0.995
        and iid_eval["mean_correct_prob"] >= ACT1_CONFIDENCE_MEAN_THRESHOLD
        and iid_eval["min_correct_prob"] >= ACT1_CONFIDENCE_MIN_THRESHOLD
    )


def train_model(
    model: torch.nn.Module,
    train_samples: list[PositionSample],
    iid_samples: list[PositionSample],
    ood_samples: list[PositionSample],
    epochs: int,
    learning_rate: float,
) -> list[dict[str, float]]:
    x_train, y_train = samples_to_tensors(train_samples)
    optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=5e-5)
    history: list[dict[str, float]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimiser.zero_grad()
        logits = model(x_train)
        loss = torch.nn.functional.cross_entropy(logits, y_train) + 0.10 * act1_margin_loss(logits, y_train)
        loss.backward()
        optimiser.step()
        train_eval = evaluate_model(model, train_samples)
        iid_eval = evaluate_model(model, iid_samples)
        ood_eval = evaluate_model(model, ood_samples)
        history.append(
            {
                "epoch": float(epoch),
                "train_loss": float(train_eval["loss"]),
                "train_accuracy": float(train_eval["accuracy"]),
                "train_mean_correct_prob": float(train_eval["mean_correct_prob"]),
                "train_min_correct_prob": float(train_eval["min_correct_prob"]),
                "iid_loss": float(iid_eval["loss"]),
                "iid_accuracy": float(iid_eval["accuracy"]),
                "iid_mean_correct_prob": float(iid_eval["mean_correct_prob"]),
                "iid_min_correct_prob": float(iid_eval["min_correct_prob"]),
                "ood_loss": float(ood_eval["loss"]),
                "ood_accuracy": float(ood_eval["accuracy"]),
            }
        )
        if epoch >= 18 and act1_confidence_ready(train_eval, iid_eval):
            break
    return history


baseline_mlp = PixelMLP()
mitigated_cnn = GapCNN()
mlp_history = train_model(
    baseline_mlp,
    biased_train_samples,
    iid_validation_samples,
    ood_audit_samples,
    epochs=80,
    learning_rate=2.5e-3,
)
cnn_history = train_model(
    mitigated_cnn,
    position_augmented_train_samples,
    position_augmented_validation_samples,
    ood_audit_samples,
    epochs=120,
    learning_rate=3.0e-3,
)


def ensure_iid_convergence() -> None:
    global baseline_mlp, mitigated_cnn, mlp_history, cnn_history
    mlp_train = evaluate_model(baseline_mlp, biased_train_samples)
    mlp_iid = evaluate_model(baseline_mlp, iid_validation_samples)
    cnn_train = evaluate_model(mitigated_cnn, position_augmented_train_samples)
    cnn_iid = evaluate_model(mitigated_cnn, position_augmented_validation_samples)
    if not act1_confidence_ready(mlp_train, mlp_iid):
        torch.manual_seed(SEED + 101)
        baseline_mlp = PixelMLP()
        mlp_history = train_model(
            baseline_mlp,
            biased_train_samples,
            iid_validation_samples,
            ood_audit_samples,
            epochs=120,
            learning_rate=2.8e-3,
        )
    if not act1_confidence_ready(cnn_train, cnn_iid):
        torch.manual_seed(SEED + 102)
        mitigated_cnn = GapCNN()
        cnn_history = train_model(
            mitigated_cnn,
            position_augmented_train_samples,
            position_augmented_validation_samples,
            ood_audit_samples,
            epochs=160,
            learning_rate=3.2e-3,
        )
    mlp_iid_final = evaluate_model(baseline_mlp, iid_validation_samples)
    cnn_iid_final = evaluate_model(mitigated_cnn, position_augmented_validation_samples)
    if not act1_confidence_ready(evaluate_model(baseline_mlp, biased_train_samples), mlp_iid_final):
        raise RuntimeError("Act I failed: Pixel MLP did not reach decisive IID confidence.")
    if not act1_confidence_ready(evaluate_model(mitigated_cnn, position_augmented_train_samples), cnn_iid_final):
        raise RuntimeError("Act I failed: CNN did not reach decisive IID confidence.")


ensure_iid_convergence()
print("Trained MLP on biased positions with decisive IID confidence.")
print("Trained CNN with shared convolution, global average pooling, position augmentation, and decisive IID confidence.")


## 4. Both models appear to work

If we only ask the ordinary IID validation question, both models look successful. This is the deliberate trap: accuracy says the task is solved before we know whether the model solved it for the right reason.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig01_both_models_appear_to_work.png" style="width:100%; border:1px solid #ddd">


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.8), constrained_layout=True)
fig.suptitle("Everything looks fine — if the test repeats the bias", fontsize=18, fontweight="bold")

card_data = [
    ("Pixel MLP", mlp_history[-1]["train_accuracy"], mlp_history[-1]["iid_accuracy"], mlp_history[-1]["iid_mean_correct_prob"], mlp_history[-1]["iid_min_correct_prob"], MODEL_COLOURS["MLP"]),
    ("CNN + augmentation", cnn_history[-1]["train_accuracy"], cnn_history[-1]["iid_accuracy"], cnn_history[-1]["iid_mean_correct_prob"], cnn_history[-1]["iid_min_correct_prob"], MODEL_COLOURS["CNN"]),
]
for axis, (model_name, train_acc, iid_acc, iid_mean_prob, iid_min_prob, colour) in zip(axes[:2], card_data, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.04, 0.08), 0.92, 0.82, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.4, transform=axis.transAxes))
    axis.text(0.10, 0.75, model_name, fontsize=15, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.10, 0.52, f"train accuracy\n{train_acc * 100:.1f}%", fontsize=14, transform=axis.transAxes)
    axis.text(0.10, 0.30, f"IID validation\n{iid_acc * 100:.1f}%", fontsize=14, fontweight="bold", transform=axis.transAxes)
    axis.text(0.10, 0.15, f"mean correct probability {iid_mean_prob:.2f}\nminimum correct probability {iid_min_prob:.2f}", fontsize=10.5, color="#555555", transform=axis.transAxes)

curve_axis = axes[2]
mlp_epochs = [row["epoch"] for row in mlp_history]
cnn_epochs = [row["epoch"] for row in cnn_history]
curve_axis.plot(mlp_epochs, [row["iid_accuracy"] for row in mlp_history], color=MODEL_COLOURS["MLP"], linewidth=2.6, label="MLP IID")
curve_axis.plot(cnn_epochs, [row["iid_accuracy"] for row in cnn_history], color=MODEL_COLOURS["CNN"], linewidth=2.6, label="CNN IID")
curve_axis.set_title("ordinary validation")
curve_axis.set_xlabel("epoch")
curve_axis.set_ylabel("accuracy")
curve_axis.set_ylim(0.0, 1.05)
curve_axis.grid(alpha=0.18)
curve_axis.legend(frameon=False, loc="lower right")
fig.text(0.5, -0.02, "At this point a normal validation report would say both models learned the task.", ha="center", fontsize=12, fontweight="bold")
save_and_show(fig, "fig01_both_models_appear_to_work.png")


## 5. The first crack: same object, different position

Now we ask a stronger behavioural question: what happens if the same object moves?

<div style="border-left:8px solid #C44E52; background:#FFF1F2; padding:18px 22px; margin:16px 0; font-size:22px; font-weight:800;">
Same moon. Same pixels. Different place. Different answer.
</div>

This is the first reveal. Accuracy said the task was solved; the counterfactual asks what evidence the model used.


In [ ]:
def group_accuracies(model: torch.nn.Module, samples: list[PositionSample]) -> dict[str, float]:
    evaluation = evaluate_model(model, samples)
    predictions = evaluation["predictions"]
    labels = evaluation["labels"]
    groups = np.array([sample.group for sample in samples])
    return {
        group: float((predictions[groups == group] == labels[groups == group]).mean())
        for group in sorted(set(groups))
    }


def subset_accuracy(model: torch.nn.Module, samples: list[PositionSample], groups: set[str]) -> float:
    selected = [sample for sample in samples if sample.group in groups]
    return float(evaluate_model(model, selected)["accuracy"])


mlp_train_eval = evaluate_model(baseline_mlp, biased_train_samples)
mlp_iid_validation_eval = evaluate_model(baseline_mlp, iid_validation_samples)
mlp_ood_eval_for_metrics = evaluate_model(baseline_mlp, ood_audit_samples)
mlp_train_accuracy = float(mlp_train_eval["accuracy"])
mlp_iid_validation_accuracy = float(mlp_iid_validation_eval["accuracy"])
mlp_iid_validation_mean_correct_prob = float(mlp_iid_validation_eval["mean_correct_prob"])
mlp_iid_validation_min_correct_prob = float(mlp_iid_validation_eval["min_correct_prob"])
mlp_ood_accuracy = float(mlp_ood_eval_for_metrics["accuracy"])
mlp_common_accuracy = subset_accuracy(baseline_mlp, ood_audit_samples, COMMON_GROUPS)
mlp_crossed_accuracy = subset_accuracy(baseline_mlp, ood_audit_samples, CROSSED_GROUPS)
mlp_uniform_accuracy = float(evaluate_model(baseline_mlp, uniform_audit_samples)["accuracy"])
mlp_group_accuracies = group_accuracies(baseline_mlp, ood_audit_samples)
mlp_worst_group_accuracy = min(mlp_group_accuracies.values())

cnn_train_eval = evaluate_model(mitigated_cnn, position_augmented_train_samples)
cnn_iid_validation_eval = evaluate_model(mitigated_cnn, position_augmented_validation_samples)
cnn_ood_eval_for_metrics = evaluate_model(mitigated_cnn, ood_audit_samples)
cnn_train_accuracy = float(cnn_train_eval["accuracy"])
cnn_iid_validation_accuracy = float(cnn_iid_validation_eval["accuracy"])
cnn_iid_validation_mean_correct_prob = float(cnn_iid_validation_eval["mean_correct_prob"])
cnn_iid_validation_min_correct_prob = float(cnn_iid_validation_eval["min_correct_prob"])
cnn_ood_accuracy = float(cnn_ood_eval_for_metrics["accuracy"])
cnn_common_accuracy = subset_accuracy(mitigated_cnn, ood_audit_samples, COMMON_GROUPS)
cnn_crossed_accuracy = subset_accuracy(mitigated_cnn, ood_audit_samples, CROSSED_GROUPS)
cnn_uniform_accuracy = float(evaluate_model(mitigated_cnn, uniform_audit_samples)["accuracy"])
cnn_group_accuracies = group_accuracies(mitigated_cnn, ood_audit_samples)
cnn_worst_group_accuracy = min(cnn_group_accuracies.values())


def fixed_pair(label: int, usual_position: str, moved_position: str, seed: int) -> tuple[PositionSample, PositionSample]:
    usual_region = LOWER_LEFT_REGION if usual_position == "lower_left" else UPPER_RIGHT_REGION
    usual_spec = make_shape_spec(label, usual_region, seed, fixed_centre=CANONICAL_CENTRES[usual_position])
    moved_spec = move_spec(usual_spec, CANONICAL_CENTRES[moved_position])
    usual = PositionSample(
        render_shape(usual_spec),
        label,
        usual_position,
        usual_spec.centre,
        group_name(label, usual_position),
        "paired counterfactual",
        seed,
        usual_spec,
    )
    moved = PositionSample(
        render_shape(moved_spec),
        label,
        moved_position,
        moved_spec.centre,
        group_name(label, moved_position),
        "paired counterfactual",
        seed,
        moved_spec,
    )
    return usual, moved


def make_counterfactual_pairs(count: int) -> list[tuple[PositionSample, PositionSample]]:
    pairs: list[tuple[PositionSample, PositionSample]] = []
    for index in range(count):
        pairs.append(fixed_pair(0, "lower_left", "upper_right", SEED + 80_000 + index))
        pairs.append(fixed_pair(1, "upper_right", "lower_left", SEED + 81_000 + index))
    return pairs


def counterfactual_flip_rate(model: torch.nn.Module, pairs: list[tuple[PositionSample, PositionSample]]) -> float:
    flips = 0
    for usual, moved in pairs:
        predictions = evaluate_model(model, [usual, moved])["predictions"]
        flips += int(predictions[0] != predictions[1])
    return flips / len(pairs)


counterfactual_pairs = make_counterfactual_pairs(30)
mlp_counterfactual_flip_rate = counterfactual_flip_rate(baseline_mlp, counterfactual_pairs)
cnn_counterfactual_flip_rate = counterfactual_flip_rate(mitigated_cnn, counterfactual_pairs)
mlp_shortcut_gap = mlp_common_accuracy - mlp_crossed_accuracy
cnn_shortcut_gap = cnn_common_accuracy - cnn_crossed_accuracy

performance_metrics = []
risk_metrics = []


### 5.1 Same-shape movement counterfactual

This is the central Clever-Hans test. We keep the object seed fixed, keep the same shape, size, rotation, brightness, and crescent or star geometry, and change only the position.

A shape-based model should keep the label when the object moves. A position-shortcut model will change its decision with the address.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/01_same_shape_movement_counterfactual.png" style="width:100%; border:1px solid #ddd">


In [ ]:
moon_lower_left, moon_upper_right = fixed_pair(0, "lower_left", "upper_right", SEED + 92_001)
star_upper_right, star_lower_left = fixed_pair(1, "upper_right", "lower_left", SEED + 92_011)
selected_counterfactual_samples = [moon_lower_left, moon_upper_right, star_upper_right, star_lower_left]
selected_mlp_eval = evaluate_model(baseline_mlp, selected_counterfactual_samples)
selected_cnn_eval = evaluate_model(mitigated_cnn, selected_counterfactual_samples)
mlp_cf_predictions = [int(value) for value in selected_mlp_eval["predictions"]]
cnn_cf_predictions = [int(value) for value in selected_cnn_eval["predictions"]]
cf_scores = {"MLP": selected_mlp_eval["star_score"], "CNN": selected_cnn_eval["star_score"]}
selected_mlp_correct_probs = [float(value) for value in selected_mlp_eval["correct_prob"]]
selected_cnn_correct_probs = [float(value) for value in selected_cnn_eval["correct_prob"]]

fig, axes = plt.subplots(2, 2, figsize=(11.4, 7.4), constrained_layout=True)
fig.suptitle("Same shape, different position, different prediction", fontsize=19, fontweight="bold")
hero_panels = [
    (axes[0, 0], "Moon lower-left", moon_lower_left, 0),
    (axes[0, 1], "Same moon upper-right", moon_upper_right, 1),
    (axes[1, 0], "Star upper-right", star_upper_right, 2),
    (axes[1, 1], "Same star lower-left", star_lower_left, 3),
]
for axis, title, sample, idx in hero_panels:
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(f"{title}\nTRUE: {CLASS_NAMES[sample.label]} | POSITION: {sample.position.replace('_', '-')}", fontsize=11.5)
    mlp_pred = mlp_cf_predictions[idx]
    cnn_pred = cnn_cf_predictions[idx]
    mlp_ok = mlp_pred == sample.label
    cnn_ok = cnn_pred == sample.label
    mlp_score = cf_scores["MLP"][idx] if mlp_pred == 1 else 1.0 - cf_scores["MLP"][idx]
    cnn_score = cf_scores["CNN"][idx] if cnn_pred == 1 else 1.0 - cf_scores["CNN"][idx]
    badge_specs = [
        (0.04, f"MLP: {CLASS_NAMES[mlp_pred]}, score {mlp_score:.2f} {'OK' if mlp_ok else 'FAIL'}", MODEL_COLOURS["MLP"], mlp_ok),
        (0.55, f"CNN: {CLASS_NAMES[cnn_pred]}, score {cnn_score:.2f} {'OK' if cnn_ok else 'FAIL'}", MODEL_COLOURS["CNN"], cnn_ok),
    ]
    for x0, text, colour, ok in badge_specs:
        axis.text(
            x0,
            -0.13,
            text,
            transform=axis.transAxes,
            fontsize=10.5,
            color="white",
            fontweight="bold",
            bbox={"boxstyle": "round,pad=0.35", "facecolor": colour if ok else "#B23A48", "edgecolor": "none", "alpha": 0.95},
        )
fig.text(0.5, -0.02, "The MLP learned where, not what.", ha="center", fontsize=14, fontweight="bold", color=MODEL_COLOURS["MLP"])
save_and_show(fig, "fig05_same_shape_moved.png")

selected_mlp_moon_counterfactual_flipped = mlp_cf_predictions[0] != mlp_cf_predictions[1]
selected_cnn_moon_counterfactual_stable = cnn_cf_predictions[0] == cnn_cf_predictions[1] == 0
selected_mlp_star_counterfactual_flipped = mlp_cf_predictions[2] != mlp_cf_predictions[3]
selected_cnn_star_counterfactual_stable = cnn_cf_predictions[2] == cnn_cf_predictions[3] == 1

selected_mlp_moon_usual_correct_prob = selected_mlp_correct_probs[0]
selected_mlp_moon_moved_wrong_prob = float(cf_scores["MLP"][1])
selected_mlp_star_usual_correct_prob = selected_mlp_correct_probs[2]
selected_mlp_star_moved_wrong_prob = float(1.0 - cf_scores["MLP"][3])
selected_cnn_moon_moved_correct_prob = selected_cnn_correct_probs[1]
selected_cnn_star_moved_correct_prob = selected_cnn_correct_probs[3]


## 6. Quantify the behavioural counterfactual

The hero panel shows the failure. The confidence path measures it: how far must the object move before the model changes its mind?

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/02_movement_confidence_paths.png" style="width:100%; border:1px solid #ddd">


In [ ]:
def sample_from_spec(spec: ShapeSpec, centre: tuple[int, int], split: str = "scan") -> PositionSample:
    moved = move_spec(spec, centre)
    position = position_name(centre)
    return PositionSample(render_shape(moved), moved.label, position, centre, group_name(moved.label, position), split, moved.seed, moved)


fixed_moon_spec = moon_lower_left.spec
fixed_star_spec = star_upper_right.spec


def interpolate_centres(start: tuple[int, int], end: tuple[int, int], steps: int = 21) -> list[tuple[int, int]]:
    centres = []
    for alpha in np.linspace(0, 1, steps):
        x = int(round((1.0 - alpha) * start[0] + alpha * end[0]))
        y = int(round((1.0 - alpha) * start[1] + alpha * end[1]))
        centres.append((x, y))
    return centres


def movement_samples(spec: ShapeSpec, start: tuple[int, int], end: tuple[int, int]) -> list[PositionSample]:
    return [sample_from_spec(spec, centre, split="movement path") for centre in interpolate_centres(start, end)]


def movement_scores(model: torch.nn.Module, spec: ShapeSpec, start: tuple[int, int], end: tuple[int, int]) -> np.ndarray:
    return evaluate_model(model, movement_samples(spec, start, end))["star_score"]


movement_progress = np.linspace(0, 1, 21)
moon_movement_samples = movement_samples(fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"])
star_movement_samples = movement_samples(fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"])
movement_path_results = {
    "moon_mlp": movement_scores(baseline_mlp, fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"]),
    "moon_cnn": movement_scores(mitigated_cnn, fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"]),
    "star_mlp": movement_scores(baseline_mlp, fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"]),
    "star_cnn": movement_scores(mitigated_cnn, fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"]),
}


def first_crossing(progress: np.ndarray, scores: np.ndarray, starts_above: bool) -> float | None:
    if starts_above:
        crossed = np.where(scores < 0.5)[0]
    else:
        crossed = np.where(scores >= 0.5)[0]
    return None if len(crossed) == 0 else float(progress[int(crossed[0])])


fig = plt.figure(figsize=(13.4, 7.6), constrained_layout=True)
grid = fig.add_gridspec(3, 10, height_ratios=[2.3, 0.95, 0.95])
fig.suptitle("How far must the object move before the model changes its mind?", fontsize=18, fontweight="bold")
for axis, title, mlp_key, cnn_key, starts_above in [
    (fig.add_subplot(grid[0, :5]), "Same moon: lower-left → upper-right", "moon_mlp", "moon_cnn", False),
    (fig.add_subplot(grid[0, 5:]), "Same star: upper-right → lower-left", "star_mlp", "star_cnn", True),
]:
    axis.plot(movement_progress, movement_path_results[mlp_key], label="MLP", color=MODEL_COLOURS["MLP"], linewidth=2.6)
    axis.plot(movement_progress, movement_path_results[cnn_key], label="CNN", color=MODEL_COLOURS["CNN"], linewidth=2.6)
    axis.axhline(0.5, color="black", linestyle="--", linewidth=1.2, label="decision threshold")
    crossing = first_crossing(movement_progress, movement_path_results[mlp_key], starts_above=starts_above)
    if crossing is not None:
        axis.axvline(crossing, color=MODEL_COLOURS["MLP"], linestyle=":", linewidth=2)
        axis.text(crossing + 0.02, 0.08, f"MLP crosses at {crossing * 100:.0f}%", color=MODEL_COLOURS["MLP"], fontsize=10)
    axis.set_title(title)
    axis.set_xlabel("movement progress")
    axis.set_ylabel("predicted probability of star")
    axis.set_ylim(-0.03, 1.03)
    axis.grid(alpha=0.18)
    axis.legend(frameon=False, loc="center right")

thumbnail_indices = [0, 5, 10, 15, 20]
for row, samples, mlp_key, cnn_key, label in [
    (1, moon_movement_samples, "moon_mlp", "moon_cnn", "moon path"),
    (2, star_movement_samples, "star_mlp", "star_cnn", "star path"),
]:
    for col, sample_index in enumerate(thumbnail_indices):
        axis = fig.add_subplot(grid[row, col * 2 : col * 2 + 2])
        sample = samples[sample_index]
        axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
        axis.set_xticks([])
        axis.set_yticks([])
        axis.set_title(
            f"{label} {movement_progress[sample_index] * 100:.0f}%\n"
            f"MLP {movement_path_results[mlp_key][sample_index]:.2f} | CNN {movement_path_results[cnn_key][sample_index]:.2f}",
            fontsize=8.5,
        )
fig.text(
    0.5,
    -0.02,
    "MLP crosses the decision boundary as the object moves. CNN remains stable on this controlled path.",
    ha="center",
    fontsize=12,
)
save_and_show(fig, "fig10_movement_path.png")


## 7. The hidden exam leak

Only now do we inspect the data-generating process. The model was not irrational. The data made position a cheap predictive rule.

Moons were usually generated in the lower-left region. Stars were usually generated in the upper-right region. IID validation repeated that pattern, so ordinary accuracy rewarded the shortcut.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/00_hidden_position_shortcut.png" style="width:100%; border:1px solid #ddd">


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.4, 7.0), constrained_layout=True)
fig.suptitle("What shortcut exists in the data?", fontsize=18, fontweight="bold")
examples = [
    (0, "lower_left", "common training pattern"),
    (0, "upper_right", "shortcut-breaking"),
    (1, "lower_left", "shortcut-breaking"),
    (1, "upper_right", "common training pattern"),
]
for axis, (label, position, status) in zip(axes.flat, examples, strict=True):
    sample = representative_sample(label, position)
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(f"{CLASS_NAMES[label].title()} at {position.replace('_', '-')}\n{status}")
    axis.set_xticks([])
    axis.set_yticks([])
    colour = "#2E8B57" if status.startswith("common") else "#C44E52"
    for spine in axis.spines.values():
        spine.set_linewidth(3.0)
        spine.set_color(colour)
fig.text(0.5, -0.02, "The true label is the shape. The hidden shortcut is where the shape appears.", ha="center", fontsize=12)
save_and_show(fig, "fig01_shape_position_grid.png")


def centres_for(samples: list[PositionSample], label: int) -> np.ndarray:
    return np.array([sample.centre for sample in samples if sample.label == label], dtype=float)


fig, axis = plt.subplots(figsize=(7.6, 7.2), constrained_layout=True)
fig.suptitle("The hidden exam leak", fontsize=18, fontweight="bold")
for region, label_text, colour in [
    (LOWER_LEFT_REGION, "moon territory", CLASS_COLOURS[0]),
    (UPPER_RIGHT_REGION, "star territory", CLASS_COLOURS[1]),
]:
    x0, x1, y0, y1 = region
    axis.fill_between([x0, x1], y0, y1, color=colour, alpha=0.14)
    axis.text((x0 + x1) / 2, (y0 + y1) / 2, label_text, ha="center", va="center", color=colour, fontweight="bold")
for label, marker in [(0, "o"), (1, "*")]:
    pts = centres_for(biased_train_samples, label)
    axis.scatter(pts[:, 0], pts[:, 1], s=42 if label == 0 else 70, alpha=0.75, marker=marker, label=f"training {CLASS_NAMES[label]} centres", color=CLASS_COLOURS[label], edgecolor="white", linewidth=0.5)
axis.set_xlim(0, IMAGE_SIZE)
axis.set_ylim(IMAGE_SIZE, 0)
axis.set_xlabel("object centre x")
axis.set_ylabel("object centre y")
axis.set_aspect("equal")
axis.legend(loc="lower right", frameon=False)
axis.set_title("Position predicts the label in the biased split, even though position is not the task.")
save_and_show(fig, "fig02_training_position_distribution.png")


## 8. Response maps: the model has learned geography

**Where does the model believe stars live?** We hold the object shape fixed and scan it across the canvas. The MLP has learned territory. The CNN is much less tied to territory.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/03_position_response_maps_with_boundaries.png" style="width:100%; border:1px solid #ddd">


In [ ]:
SCAN_VALUES = np.linspace(22, 106, 17).astype(int)


def position_response(model: torch.nn.Module, spec: ShapeSpec) -> np.ndarray:
    samples = [sample_from_spec(spec, (int(x), int(y))) for y in SCAN_VALUES for x in SCAN_VALUES]
    scores = evaluate_model(model, samples)["star_score"]
    return scores.reshape(len(SCAN_VALUES), len(SCAN_VALUES))


moon_position_response_mlp = position_response(baseline_mlp, fixed_moon_spec)
star_position_response_mlp = position_response(baseline_mlp, fixed_star_spec)
moon_position_response_cnn = position_response(mitigated_cnn, fixed_moon_spec)
star_position_response_cnn = position_response(mitigated_cnn, fixed_star_spec)


def sensitivity_index(matrix: np.ndarray) -> float:
    return float(np.max(matrix) - np.min(matrix))


def shape_consistency(matrix: np.ndarray, true_label: int) -> float:
    predictions = (matrix >= 0.5).astype(int)
    return float(np.mean(predictions == true_label))


position_response_metrics = {
    "mlp_moon_psi": sensitivity_index(moon_position_response_mlp),
    "mlp_star_psi": sensitivity_index(star_position_response_mlp),
    "cnn_moon_psi": sensitivity_index(moon_position_response_cnn),
    "cnn_star_psi": sensitivity_index(star_position_response_cnn),
    "mlp_moon_shape_consistency": shape_consistency(moon_position_response_mlp, 0),
    "mlp_star_shape_consistency": shape_consistency(star_position_response_mlp, 1),
    "cnn_moon_shape_consistency": shape_consistency(moon_position_response_cnn, 0),
    "cnn_star_shape_consistency": shape_consistency(star_position_response_cnn, 1),
}
mlp_position_sensitivity = float(np.mean([position_response_metrics["mlp_moon_psi"], position_response_metrics["mlp_star_psi"]]))
cnn_position_sensitivity = float(np.mean([position_response_metrics["cnn_moon_psi"], position_response_metrics["cnn_star_psi"]]))
mlp_shape_consistency = float(np.mean([position_response_metrics["mlp_moon_shape_consistency"], position_response_metrics["mlp_star_shape_consistency"]]))
cnn_shape_consistency = float(np.mean([position_response_metrics["cnn_moon_shape_consistency"], position_response_metrics["cnn_star_shape_consistency"]]))

response_panels = [
    ("MLP fixed moon", moon_position_response_mlp, 0, "star score rises\nin upper-right"),
    ("MLP fixed star", star_position_response_mlp, 1, "star score falls\nin lower-left"),
    ("CNN fixed moon", moon_position_response_cnn, 0, "stable moon\nprediction"),
    ("CNN fixed star", star_position_response_cnn, 1, "stable star\nprediction"),
]
fig, axes = plt.subplots(2, 2, figsize=(11.4, 9.2), constrained_layout=True)
fig.suptitle("What happens when we move the same object around the image?", fontsize=18, fontweight="bold")
for axis, (title, matrix, true_label, annotation) in zip(axes.flat, response_panels, strict=True):
    im = axis.imshow(
        matrix,
        cmap="RdYlBu_r",
        vmin=0,
        vmax=1,
        extent=[SCAN_VALUES[0], SCAN_VALUES[-1], SCAN_VALUES[-1], SCAN_VALUES[0]],
        interpolation="bicubic",
    )
    axis.contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.5], colors="black", linewidths=1.4)
    axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[2]), LOWER_LEFT_REGION[1]-LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[3]-LOWER_LEFT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[0], linewidth=2))
    axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[2]), UPPER_RIGHT_REGION[1]-UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[3]-UPPER_RIGHT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[1], linewidth=2))
    psi = sensitivity_index(matrix)
    consistency = shape_consistency(matrix, true_label)
    axis.set_title(f"{title}\nPSI {psi:.2f}; shape consistency {consistency * 100:.0f}%")
    axis.text(0.04, 0.08, annotation, transform=axis.transAxes, fontsize=10.5, fontweight="bold", color="white", bbox={"boxstyle":"round,pad=0.35", "facecolor":"black", "alpha":0.70, "edgecolor":"none"})
    axis.set_xlabel("object centre x")
    axis.set_ylabel("object centre y")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, label="predicted probability of star")
fig.text(0.18, 0.015, "blue = moon-like score", color="#2D6F9F", fontsize=11, fontweight="bold")
fig.text(0.38, 0.015, "red = star-like score", color="#B23A48", fontsize=11, fontweight="bold")
fig.text(0.66, 0.015, "PSI is score range; shape consistency is the share of positions classified as the true shape.", fontsize=10.5)
save_and_show(fig, "fig06_position_response_maps.png")


### 8.1 Decision boundary in position space

The boundary view is the same question in a more abstract form: does the decision boundary cut the canvas by location, or stay stable for a fixed shape?


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.4, 9.0), constrained_layout=True)
fig.suptitle("Is the decision boundary about shape or position?", fontsize=18, fontweight="bold")
boundary_panels = [
    ("MLP: same moon", moon_position_response_mlp, moon_lower_left.centre, moon_upper_right.centre),
    ("CNN: same moon", moon_position_response_cnn, moon_lower_left.centre, moon_upper_right.centre),
    ("MLP: same star", star_position_response_mlp, star_upper_right.centre, star_lower_left.centre),
    ("CNN: same star", star_position_response_cnn, star_upper_right.centre, star_lower_left.centre),
]
decision_boundary_results = {
    "mlp_moon_crosses": bool(np.any(moon_position_response_mlp >= 0.5) and np.any(moon_position_response_mlp < 0.5)),
    "mlp_star_crosses": bool(np.any(star_position_response_mlp >= 0.5) and np.any(star_position_response_mlp < 0.5)),
    "cnn_moon_consistency": shape_consistency(moon_position_response_cnn, 0),
    "cnn_star_consistency": shape_consistency(star_position_response_cnn, 1),
}
for axis, (title, matrix, start_centre, end_centre) in zip(axes.flat, boundary_panels, strict=True):
    axis.contourf(SCAN_VALUES, SCAN_VALUES, matrix, levels=np.linspace(0, 1, 15), cmap="RdYlBu_r", alpha=0.92)
    axis.contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.5], colors="black", linewidths=1.8)
    axis.annotate("", xy=end_centre, xytext=start_centre, arrowprops={"arrowstyle":"->", "color":"black", "lw":2.2, "linestyle":"--"})
    axis.scatter([start_centre[0]], [start_centre[1]], s=75, color="#2E8B57", zorder=5)
    axis.scatter([end_centre[0]], [end_centre[1]], s=75, color="#C44E52", zorder=5)
    axis.text(start_centre[0] + 2, start_centre[1] + 4, "usual", color="#2E8B57", fontweight="bold")
    axis.text(end_centre[0] + 2, end_centre[1] - 4, "moved", color="#C44E52", fontweight="bold")
    axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[2]), LOWER_LEFT_REGION[1]-LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[3]-LOWER_LEFT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[0], linewidth=2))
    axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[2]), UPPER_RIGHT_REGION[1]-UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[3]-UPPER_RIGHT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[1], linewidth=2))
    axis.set_title(title)
    axis.set_xlim(SCAN_VALUES[0], SCAN_VALUES[-1])
    axis.set_ylim(SCAN_VALUES[-1], SCAN_VALUES[0])
    axis.set_xlabel("object centre x")
    axis.set_ylabel("object centre y")
fig.text(0.5, -0.02, "For the MLP, the boundary cuts the canvas by location. For the CNN, the decision is much more stable for the same shape across positions.", ha="center", fontsize=12)
save_and_show(fig, "fig07_position_decision_boundary.png")


## 9. Shape versus position

The movement counterfactual asks what happens when shape is fixed and position changes. The morph probe asks the complementary question: when position is fixed, does confidence follow the changing shape?

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/04_shape_position_score_surface.png" style="width:100%; border:1px solid #ddd">


In [ ]:
def render_morph_image(alpha: float, centre: tuple[int, int], seed: int) -> Image.Image:
    """Render a controlled shape morph without making a dim pixel cross-fade.

    The previous version linearly blended two rendered images. Mid-morph frames
    contained a grey ghost of both shapes, so the CNN was being asked to
    classify an object unlike the training distribution. This version morphs
    the foreground mask while keeping the object bright and the address fixed.
    """
    alpha = float(np.clip(alpha, 0.0, 1.0))
    moon_spec = make_shape_spec(0, FREE_REGION, seed, fixed_centre=centre)
    star_spec = ShapeSpec(
        label=1,
        centre=centre,
        seed=seed,
        size=moon_spec.size,
        rotation=moon_spec.rotation + 8.0,
        brightness=moon_spec.brightness,
        stroke=moon_spec.stroke,
        crescent=moon_spec.crescent,
    )
    local_rng = rng_for(seed + 99)
    yy, xx = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE]
    texture = (
        38.0
        + 1.8 * np.sin(xx / 11.0)
        + 1.3 * np.cos(yy / 13.0)
        + local_rng.normal(0.0, 1.3, (IMAGE_SIZE, IMAGE_SIZE))
    )
    moon_mask = np.asarray(object_mask(moon_spec), dtype=np.float32) / 255.0
    star_mask = np.asarray(object_mask(star_spec), dtype=np.float32) / 255.0

    # Non-linear weights suppress the fading ghost of the outgoing shape while
    # still giving a visible ambiguous middle frame.
    moon_weight = (1.0 - alpha) ** 3
    star_weight = alpha ** 3
    combined = np.maximum(moon_weight * moon_mask, star_weight * star_mask)
    if float(combined.max()) > 1e-8:
        combined = combined / float(combined.max())
    foreground_alpha = np.clip((combined - 0.12) / 0.58, 0.0, 1.0)
    image = texture * (1.0 - foreground_alpha) + float(moon_spec.brightness) * foreground_alpha
    return Image.fromarray(np.clip(image, 0, 255).astype(np.uint8), "L")

def score_image(model: torch.nn.Module, image: Image.Image) -> float:
    """Score a generated diagnostic image without using the sample tensor cache."""
    array = np.asarray(image.convert("L").resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
    x = torch.tensor(array[None, None, :, :], dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        return float(torch.softmax(model(x), dim=1)[0, 1].item())


morph_alphas = np.linspace(0, 1, 21)
strip_alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
position_alphas = np.linspace(0, 1, 17)

shape_morph_results: dict[str, dict[str, np.ndarray]] = {}
for position_name_key, centre in CANONICAL_CENTRES.items():
    morph_images = [render_morph_image(float(alpha), centre, SEED + 93_000) for alpha in morph_alphas]
    shape_morph_results[position_name_key] = {
        "mlp": np.array([score_image(baseline_mlp, image) for image in morph_images]),
        "cnn": np.array([score_image(mitigated_cnn, image) for image in morph_images]),
    }

fig, axes = plt.subplots(2, 5, figsize=(13.2, 6.2), constrained_layout=True)
fig.suptitle("Does the score follow shape or position?", fontsize=18, fontweight="bold")
for row, (position_name_key, row_title) in enumerate([("lower_left", "lower-left"), ("upper_right", "upper-right")]):
    centre = CANONICAL_CENTRES[position_name_key]
    for col, alpha in enumerate(strip_alphas):
        axis = axes[row, col]
        image = render_morph_image(alpha, centre, SEED + 93_000)
        nearest_index = int(round(alpha * (len(morph_alphas) - 1)))
        mlp_score = shape_morph_results[position_name_key]["mlp"][nearest_index]
        cnn_score = shape_morph_results[position_name_key]["cnn"][nearest_index]
        axis.imshow(image, cmap="gray", vmin=0, vmax=255)
        axis.set_xticks([])
        axis.set_yticks([])
        if row == 0:
            axis.set_title(["moon", "moon-ish", "ambiguous", "star-ish", "star"][col], fontsize=11)
        axis.text(
            0.5,
            -0.11,
            f"{row_title}\nMLP star {mlp_score:.2f}\nCNN star {cnn_score:.2f}",
            transform=axis.transAxes,
            ha="center",
            va="top",
            fontsize=9.5,
            bbox={"boxstyle":"round,pad=0.25", "facecolor":"white", "edgecolor":"#DDDDDD", "alpha":0.95},
        )
fig.text(0.5, -0.03, "At upper-right, the MLP can score an object as star-like before it looks star-like. The CNN tracks the morph more consistently.", ha="center", fontsize=12, fontweight="bold")
save_and_show(fig, "fig08_shape_morph_strip.png")

shape_position_surface_mlp = np.zeros((len(morph_alphas), len(position_alphas)))
shape_position_surface_cnn = np.zeros_like(shape_position_surface_mlp)
for row, shape_alpha in enumerate(morph_alphas):
    for col, position_alpha in enumerate(position_alphas):
        x = int((1.0 - position_alpha) * CANONICAL_CENTRES["lower_left"][0] + position_alpha * CANONICAL_CENTRES["upper_right"][0])
        y = int((1.0 - position_alpha) * CANONICAL_CENTRES["lower_left"][1] + position_alpha * CANONICAL_CENTRES["upper_right"][1])
        morph_image = render_morph_image(float(shape_alpha), (x, y), SEED + 93_000)
        shape_position_surface_mlp[row, col] = score_image(baseline_mlp, morph_image)
        shape_position_surface_cnn[row, col] = score_image(mitigated_cnn, morph_image)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.4), constrained_layout=True)
fig.suptitle("Shape versus position: what controls the score?", fontsize=18, fontweight="bold")
for axis, title, surface, note in [
    (axes[0], "MLP", shape_position_surface_mlp, "position dominates"),
    (axes[1], "CNN", shape_position_surface_cnn, "shape dominates"),
]:
    im = axis.imshow(surface, cmap="RdYlBu_r", vmin=0, vmax=1, origin="lower", aspect="auto", extent=[0, 1, 0, 1], interpolation="bicubic")
    axis.contour(position_alphas, morph_alphas, surface, levels=[0.5], colors="black", linewidths=1.5)
    axis.text(0.04, 0.88, note, transform=axis.transAxes, fontsize=12, fontweight="bold", color="white", bbox={"boxstyle":"round,pad=0.35", "facecolor":"black", "alpha":0.72, "edgecolor":"none"})
    axis.set_title(title)
    axis.set_xlabel("position: lower-left → upper-right")
    axis.set_ylabel("shape: moon → star")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.88, label="predicted probability of star")
fig.text(0.5, -0.03, "A vertical boundary means position controls the score. A horizontal boundary means shape controls the score.", ha="center", fontsize=12)
save_and_show(fig, "fig09_shape_position_surface.png")


## 10. Heatmaps are not enough

A saliency map can highlight object pixels even when the model is using position. The key lesson is not that heatmaps are useless; it is that they are incomplete without behavioural counterfactuals.

Saliency may highlight the object, but behavioural probes reveal that the object’s address controls the decision.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/05_why_heatmaps_are_not_enough.png" style="width:100%; border:1px solid #ddd">


In [ ]:
def tensor_for_image(image: Image.Image) -> torch.Tensor:
    array = np.asarray(image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
    return torch.tensor(array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)


def smoothgrad_saliency(
    model: torch.nn.Module,
    image: Image.Image,
    target_class: int,
    samples: int = 4,
    noise: float = 0.035,
) -> np.ndarray:
    model.eval()
    base = tensor_for_image(image)
    saliency = torch.zeros_like(base)
    for _ in range(samples):
        noisy = (base + torch.randn_like(base) * noise).clamp(0, 1).requires_grad_(True)
        score = model(noisy)[0, target_class]
        model.zero_grad(set_to_none=True)
        score.backward()
        saliency += noisy.grad.detach().abs()
    saliency = saliency / samples
    saliency_array = saliency.squeeze().cpu().numpy()
    return saliency_array / max(float(saliency_array.max()), 1e-8)


moved_moon_failure = moon_upper_right
moved_star_failure = star_lower_left
torch.manual_seed(SEED + 601)
saliency_cases = [
    ("MLP moved moon failure", baseline_mlp, moved_moon_failure, int(mlp_cf_predictions[1])),
    ("CNN moved moon success", mitigated_cnn, moved_moon_failure, int(cnn_cf_predictions[1])),
    ("MLP moved star failure", baseline_mlp, moved_star_failure, int(mlp_cf_predictions[3])),
    ("CNN moved star success", mitigated_cnn, moved_star_failure, int(cnn_cf_predictions[3])),
]
saliency_maps = [
    (title, sample, smoothgrad_saliency(model, sample.image, target), model)
    for title, model, sample, target in saliency_cases
]


def scaled_region_mask(region: tuple[int, int, int, int]) -> np.ndarray:
    scale = MODEL_SIZE / IMAGE_SIZE
    x0, x1, y0, y1 = region
    mask = np.zeros((MODEL_SIZE, MODEL_SIZE), dtype=bool)
    mask[int(y0 * scale) : int(y1 * scale), int(x0 * scale) : int(x1 * scale)] = True
    return mask


def resize_object_mask(sample: PositionSample) -> np.ndarray:
    return np.asarray(
        object_mask(sample.spec).resize((MODEL_SIZE, MODEL_SIZE), Image.Resampling.NEAREST),
        dtype=np.float32,
    ) > 0


def attribution_density_records(saliency: np.ndarray, sample: PositionSample) -> dict[str, dict[str, float]]:
    object_region = resize_object_mask(sample)
    usual_region = scaled_region_mask(LOWER_LEFT_REGION if sample.label == 0 else UPPER_RIGHT_REGION)
    regions = {
        "object": object_region,
        "usual position": usual_region,
        "off object": ~object_region,
    }
    total = float(saliency.sum()) + 1e-8
    records: dict[str, dict[str, float]] = {}
    for name, mask in regions.items():
        attribution_share = float(saliency[mask].sum() / total)
        area_share = float(mask.mean())
        records[name] = {
            "attribution_share": attribution_share,
            "area_share": area_share,
            "density": float(attribution_share / max(area_share, 1e-8)),
        }
    return records


attribution_density_results = {
    title: attribution_density_records(saliency, sample)
    for title, sample, saliency, _model in saliency_maps
}
mlp_object_mass = attribution_density_results["MLP moved moon failure"]["object"]["attribution_share"]

fig = plt.figure(figsize=(14.6, 9.2), constrained_layout=True)
grid = fig.add_gridspec(3, 4, height_ratios=[1.0, 1.0, 1.12])
fig.suptitle("What does a heatmap show — and what does it miss?", fontsize=18, fontweight="bold")
for col, (title, sample, saliency, _model) in enumerate(saliency_maps):
    image_axis = fig.add_subplot(grid[0, col])
    image_axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    image_axis.set_title(title, fontsize=10.5)
    image_axis.set_xticks([])
    image_axis.set_yticks([])

    saliency_axis = fig.add_subplot(grid[1, col])
    saliency_axis.imshow(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), cmap="gray", alpha=0.55)
    saliency_axis.imshow(saliency, cmap="magma", alpha=0.64)
    saliency_axis.set_xticks([])
    saliency_axis.set_yticks([])
    saliency_axis.set_title("SmoothGrad", fontsize=10)

share_axis = fig.add_subplot(grid[2, :2])
density_axis = fig.add_subplot(grid[2, 2:])
case_labels = [title.replace(" moved ", "\nmoved ").replace(" failure", "\nfailure").replace(" success", "\nsuccess") for title in attribution_density_results]
region_names = ["object", "usual position", "off object"]
region_colours = ["#4C78A8", "#F2A541", "#999999"]
x = np.arange(len(case_labels))
width = 0.22
for offset, region, colour in zip([-width, 0.0, width], region_names, region_colours, strict=True):
    share_axis.bar(
        x + offset,
        [attribution_density_results[title][region]["attribution_share"] for title in attribution_density_results],
        width,
        label=region,
        color=colour,
    )
    density_axis.bar(
        x + offset,
        [attribution_density_results[title][region]["density"] for title in attribution_density_results],
        width,
        label=region,
        color=colour,
    )
for axis, ylabel, title in [
    (share_axis, "share of attribution mass", "Raw attribution share"),
    (density_axis, "attribution share / area share", "Area-normalised relevance density"),
]:
    axis.set_xticks(x)
    axis.set_xticklabels(case_labels, fontsize=8.2)
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.18)
    axis.legend(frameon=False, fontsize=8.5)
fig.text(
    0.5,
    -0.015,
    "Saliency can highlight object pixels and still miss the shortcut. Area normalisation helps, but movement counterfactuals remain the stronger test.",
    ha="center",
    fontsize=11.5,
    fontweight="bold",
)
save_and_show(fig, "fig11_saliency_comparison.png")


def predicted_class(model: torch.nn.Module, sample: PositionSample) -> int:
    return int(evaluate_model(model, [sample])["predictions"][0])


def average_saliency_for_samples(
    model: torch.nn.Module,
    samples: list[PositionSample],
    max_examples: int,
    seed_offset: int,
) -> np.ndarray:
    selected = samples[:max_examples]
    if not selected:
        return np.zeros((MODEL_SIZE, MODEL_SIZE), dtype=np.float32)
    torch.manual_seed(SEED + seed_offset)
    maps = []
    for sample in selected:
        target = predicted_class(model, sample)
        saliency = smoothgrad_saliency(model, sample.image, target, samples=2, noise=0.03)
        maps.append(saliency / (float(saliency.sum()) + 1e-8))
    average = np.mean(np.stack(maps), axis=0)
    return average / max(float(average.max()), 1e-8)


mlp_ood_eval = evaluate_model(baseline_mlp, ood_audit_samples)
cnn_ood_eval = evaluate_model(mitigated_cnn, ood_audit_samples)
mlp_iid_eval = evaluate_model(baseline_mlp, iid_validation_samples)
mlp_ood_failures = [sample for sample, pred in zip(ood_audit_samples, mlp_ood_eval["predictions"], strict=True) if int(pred) != sample.label]
cnn_ood_correct = [sample for sample, pred in zip(ood_audit_samples, cnn_ood_eval["predictions"], strict=True) if int(pred) == sample.label]
mlp_iid_correct = [sample for sample, pred in zip(iid_validation_samples, mlp_iid_eval["predictions"], strict=True) if int(pred) == sample.label]

average_relevance_results = {
    "MLP OOD failures": average_saliency_for_samples(baseline_mlp, mlp_ood_failures, max_examples=4, seed_offset=701),
    "CNN correct OOD": average_saliency_for_samples(mitigated_cnn, cnn_ood_correct, max_examples=4, seed_offset=702),
    "MLP correct IID": average_saliency_for_samples(baseline_mlp, mlp_iid_correct, max_examples=4, seed_offset=703),
}

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.8), constrained_layout=True)
fig.suptitle("Where does each model tend to place relevance?", fontsize=18, fontweight="bold")
for axis, (title, relevance) in zip(axes, average_relevance_results.items(), strict=True):
    im = axis.imshow(relevance, cmap="magma", vmin=0, vmax=1, interpolation="bilinear")
    axis.set_title(title)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0] * MODEL_SIZE / IMAGE_SIZE, LOWER_LEFT_REGION[2] * MODEL_SIZE / IMAGE_SIZE), (LOWER_LEFT_REGION[1] - LOWER_LEFT_REGION[0]) * MODEL_SIZE / IMAGE_SIZE, (LOWER_LEFT_REGION[3] - LOWER_LEFT_REGION[2]) * MODEL_SIZE / IMAGE_SIZE, fill=False, edgecolor=CLASS_COLOURS[0], linewidth=1.4))
    axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0] * MODEL_SIZE / IMAGE_SIZE, UPPER_RIGHT_REGION[2] * MODEL_SIZE / IMAGE_SIZE), (UPPER_RIGHT_REGION[1] - UPPER_RIGHT_REGION[0]) * MODEL_SIZE / IMAGE_SIZE, (UPPER_RIGHT_REGION[3] - UPPER_RIGHT_REGION[2]) * MODEL_SIZE / IMAGE_SIZE, fill=False, edgecolor=CLASS_COLOURS[1], linewidth=1.4))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.72, label="normalised average relevance")
fig.text(0.5, -0.02, "Average relevance is a global diagnostic. It supports the audit only when read with the movement and response-map evidence.", ha="center", fontsize=11.5)
save_and_show(fig, "fig12_average_relevance_maps.png")


def penultimate_features(model: torch.nn.Module, samples: list[PositionSample]) -> np.ndarray:
    x, _ = samples_to_tensors(samples)
    model.eval()
    with torch.no_grad():
        if isinstance(model, PixelMLP):
            features = model.net[:-1](x)
        elif isinstance(model, GapCNN):
            features = model.features(x).flatten(1)
        else:
            raise TypeError(f"Unsupported model type: {type(model)}")
    return features.cpu().numpy().astype(np.float32)


def nearest_neighbours(
    query_features: np.ndarray,
    train_features: np.ndarray,
    train_samples: list[PositionSample],
    k: int = 4,
) -> list[list[tuple[PositionSample, float]]]:
    train_norm = train_features / np.maximum(np.linalg.norm(train_features, axis=1, keepdims=True), 1e-8)
    query_norm = query_features / np.maximum(np.linalg.norm(query_features, axis=1, keepdims=True), 1e-8)
    distances = 1.0 - query_norm @ train_norm.T
    neighbour_sets: list[list[tuple[PositionSample, float]]] = []
    for row in distances:
        order = np.argsort(row)[:k]
        neighbour_sets.append([(train_samples[int(index)], float(row[int(index)])) for index in order])
    return neighbour_sets


representation_queries = [moved_moon_failure, moved_star_failure]
representation_query_labels = ["moved moon failure", "moved star failure"]
mlp_train_features = penultimate_features(baseline_mlp, biased_train_samples)
cnn_train_features = penultimate_features(mitigated_cnn, position_augmented_train_samples)
mlp_query_features = penultimate_features(baseline_mlp, representation_queries)
cnn_query_features = penultimate_features(mitigated_cnn, representation_queries)
representation_neighbour_results = {
    "MLP": nearest_neighbours(mlp_query_features, mlp_train_features, biased_train_samples, k=4),
    "CNN": nearest_neighbours(cnn_query_features, cnn_train_features, position_augmented_train_samples, k=4),
}

fig = plt.figure(figsize=(14.2, 8.4), constrained_layout=True)
grid = fig.add_gridspec(4, 6, width_ratios=[1.18, 1, 1, 1, 1, 0.18])
fig.suptitle("What does the representation think this looks like?", fontsize=18, fontweight="bold")
for row, (query, query_label) in enumerate(zip(representation_queries, representation_query_labels, strict=True)):
    query_axis = fig.add_subplot(grid[row * 2 : row * 2 + 2, 0])
    query_axis.imshow(query.image, cmap="gray", vmin=0, vmax=255)
    query_axis.set_xticks([])
    query_axis.set_yticks([])
    query_axis.set_title(f"Query\n{query_label}\ntrue {CLASS_NAMES[query.label]} | {query.position.replace('_', '-')}", fontsize=10)
    for model_col, model_name in enumerate(["MLP", "CNN"]):
        for index, (neighbour, distance) in enumerate(representation_neighbour_results[model_name][row]):
            axis = fig.add_subplot(grid[row * 2 + model_col, index + 1])
            axis.imshow(neighbour.image, cmap="gray", vmin=0, vmax=255)
            axis.set_xticks([])
            axis.set_yticks([])
            axis.set_title(
                f"{model_name} nn {index + 1}\n{CLASS_NAMES[neighbour.label]} | {neighbour.position.replace('_', '-')}\nd={distance:.2f}",
                fontsize=8.5,
            )
            for spine in axis.spines.values():
                spine.set_edgecolor(MODEL_COLOURS[model_name])
                spine.set_linewidth(1.8)
fig.text(0.5, -0.015, "Nearest neighbours show what the learned feature space treats as similar. They are provenance-style evidence, not causal proof.", ha="center", fontsize=11.5, fontweight="bold")
save_and_show(fig, "fig13_representation_neighbours.png")


def standardise_features(train: np.ndarray, test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = train.mean(axis=0, keepdims=True)
    std = train.std(axis=0, keepdims=True) + 1e-6
    return (train - mean) / std, (test - mean) / std


def train_probe(features: np.ndarray, labels: np.ndarray, seed_offset: int) -> float:
    rng = np.random.default_rng(SEED + seed_offset)
    order = rng.permutation(len(labels))
    split = int(len(labels) * 0.7)
    train_idx = order[:split]
    test_idx = order[split:]
    x_train_np, x_test_np = standardise_features(features[train_idx], features[test_idx])
    x_train = torch.tensor(x_train_np, dtype=torch.float32)
    y_train = torch.tensor(labels[train_idx], dtype=torch.long)
    x_test = torch.tensor(x_test_np, dtype=torch.float32)
    y_test = torch.tensor(labels[test_idx], dtype=torch.long)
    torch.manual_seed(SEED + seed_offset)
    probe = torch.nn.Linear(x_train.shape[1], 2)
    optimiser = torch.optim.Adam(probe.parameters(), lr=0.05, weight_decay=1e-3)
    for _ in range(160):
        optimiser.zero_grad()
        loss = torch.nn.functional.cross_entropy(probe(x_train), y_train)
        loss.backward()
        optimiser.step()
    with torch.no_grad():
        predictions = probe(x_test).argmax(dim=1)
    return float((predictions == y_test).float().mean().item())


probe_samples = ood_audit_samples
shape_probe_labels = np.array([sample.label for sample in probe_samples], dtype=np.int64)
position_probe_labels = np.array([1 if sample.position == "upper_right" else 0 for sample in probe_samples], dtype=np.int64)
mlp_probe_features = penultimate_features(baseline_mlp, probe_samples)
cnn_probe_features = penultimate_features(mitigated_cnn, probe_samples)
representation_probe_results = {
    "MLP": {
        "shape": train_probe(mlp_probe_features, shape_probe_labels, seed_offset=801),
        "position": train_probe(mlp_probe_features, position_probe_labels, seed_offset=802),
    },
    "CNN": {
        "shape": train_probe(cnn_probe_features, shape_probe_labels, seed_offset=803),
        "position": train_probe(cnn_probe_features, position_probe_labels, seed_offset=804),
    },
}

fig, axis = plt.subplots(figsize=(9.8, 5.8), constrained_layout=True)
fig.suptitle("What is easier to read from the representation?", fontsize=18, fontweight="bold")
probe_metrics = ["shape", "position"]
x = np.arange(len(probe_metrics))
width = 0.34
for offset, model_name in [(-width / 2, "MLP"), (width / 2, "CNN")]:
    values = [representation_probe_results[model_name][metric] for metric in probe_metrics]
    bars = axis.bar(x + offset, values, width, label=model_name, color=MODEL_COLOURS[model_name])
    for bar, value in zip(bars, values, strict=True):
        axis.text(bar.get_x() + bar.get_width() / 2, value + 0.025, f"{value * 100:.0f}%", ha="center", fontsize=10, fontweight="bold")
axis.set_xticks(x)
axis.set_xticklabels(["shape label", "position group"])
axis.set_ylim(0, 1.08)
axis.set_ylabel("linear probe accuracy on held-out audit examples")
axis.grid(axis="y", alpha=0.18)
axis.legend(frameon=False, loc="lower right")
fig.text(0.5, -0.025, "A probe asks what information is easy to decode from frozen penultimate features. It is diagnostic, not a proof of causal mechanism.", ha="center", fontsize=11.5)
save_and_show(fig, "fig14_representation_probes.png")


def mask_patch(image: Image.Image, box: tuple[int, int, int, int]) -> Image.Image:
    edited = image.copy()
    draw = ImageDraw.Draw(edited)
    scale = IMAGE_SIZE / MODEL_SIZE
    scaled_box = tuple(int(value * scale) for value in box)
    draw.rectangle(scaled_box, fill=38)
    return edited


def patch_boxes(grid: int = 8) -> list[tuple[int, int, int, int]]:
    step = MODEL_SIZE // grid
    boxes = []
    for y in range(0, MODEL_SIZE, step):
        for x in range(0, MODEL_SIZE, step):
            boxes.append((x, y, min(x + step, MODEL_SIZE), min(y + step, MODEL_SIZE)))
    return boxes


def wrong_class_score(model: torch.nn.Module, image: Image.Image, wrong_class: int) -> float:
    sample = PositionSample(image, 0, "diagnostic", (0, 0), "diagnostic", "diagnostic", SEED, moved_moon_failure.spec)
    evaluation = evaluate_model(model, [sample])
    if wrong_class == 1:
        return float(evaluation["star_score"][0])
    return float(1.0 - evaluation["star_score"][0])


wrong_class_for_moved_moon = int(mlp_cf_predictions[1])
original_wrong_score = wrong_class_score(baseline_mlp, moved_moon_failure.image, wrong_class_for_moved_moon)
patch_effects = []
for box in patch_boxes():
    score = wrong_class_score(baseline_mlp, mask_patch(moved_moon_failure.image, box), wrong_class_for_moved_moon)
    patch_effects.append((box, original_wrong_score - score))
ranked_patches = [box for box, _ in sorted(patch_effects, key=lambda item: item[1], reverse=True)]
control_patches = [box for box, _ in sorted(patch_effects, key=lambda item: item[1])]


def mask_many(image: Image.Image, boxes: list[tuple[int, int, int, int]], k: int) -> Image.Image:
    edited = image.copy()
    for box in boxes[:k]:
        edited = mask_patch(edited, box)
    return edited


minimal_masking_results: list[dict[str, Any]] = []
for k in [0, 1, 3, 5, 8, 12]:
    ranked_image = mask_many(moved_moon_failure.image, ranked_patches, k)
    control_image = mask_many(moved_moon_failure.image, control_patches, k)
    minimal_masking_results.append(
        {
            "patches_hidden": k,
            "ranked_score": wrong_class_score(baseline_mlp, ranked_image, wrong_class_for_moved_moon),
            "control_score": wrong_class_score(baseline_mlp, control_image, wrong_class_for_moved_moon),
            "ranked_image": ranked_image,
            "control_image": control_image,
        }
    )
best_masking_result = min(minimal_masking_results, key=lambda row: row["ranked_score"])
best_patch_drop = float(original_wrong_score - best_masking_result["ranked_score"])

fig = plt.figure(figsize=(12.8, 6.8), constrained_layout=True)
grid = fig.add_gridspec(2, 4, height_ratios=[1.0, 1.2])
fig.suptitle("How little evidence must be hidden to weaken the wrong decision?", fontsize=18, fontweight="bold")
image_panels = [
    ("Original moved moon", moved_moon_failure.image, original_wrong_score, 0),
    ("Top-ranked patches hidden", best_masking_result["ranked_image"], best_masking_result["ranked_score"], best_masking_result["patches_hidden"]),
    ("Low-importance control", best_masking_result["control_image"], best_masking_result["control_score"], best_masking_result["patches_hidden"]),
]
for col, (title, image, score, hidden) in enumerate(image_panels):
    axis = fig.add_subplot(grid[0, col])
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(f"{title}\nwrong score {score:.2f}; hidden {hidden}", fontsize=10)
verdict_axis = fig.add_subplot(grid[0, 3])
verdict_axis.axis("off")
verdict = "The prediction flips on the ranked path." if best_masking_result["ranked_score"] < 0.5 else f"The prediction does not flip, but the wrong-class score drops by {best_patch_drop * 100:.1f}pp."
verdict_axis.text(0.0, 0.62, verdict, fontsize=12, fontweight="bold", wrap=True)
verdict_axis.text(0.0, 0.28, "Patch masking is diagnostic evidence. In this position-shortcut demo, movement remains the stronger counterfactual.", fontsize=10.5, wrap=True)
path_axis = fig.add_subplot(grid[1, :])
ks = [row["patches_hidden"] for row in minimal_masking_results]
path_axis.plot(ks, [row["ranked_score"] for row in minimal_masking_results], marker="o", linewidth=2.6, label="top-ranked patches", color=MODEL_COLOURS["MLP"])
path_axis.plot(ks, [row["control_score"] for row in minimal_masking_results], marker="o", linewidth=2.1, label="low-importance control", color="#777777")
path_axis.axhline(0.5, color="black", linestyle="--", linewidth=1.0, label="decision threshold")
path_axis.set_xlabel("patches hidden")
path_axis.set_ylabel("wrong-class model score")
path_axis.set_ylim(0.0, 1.03)
path_axis.grid(alpha=0.18)
path_axis.legend(frameon=False, loc="best")
save_and_show(fig, "fig15_minimal_evidence_removal.png")

fig, axis = plt.subplots(figsize=(12.8, 6.4), constrained_layout=True)
axis.axis("off")
fig.suptitle("What changes the model's decision?", fontsize=18, fontweight="bold")
summary_cells = [
    ("Move object", "MLP", "same object\nflips", MODEL_COLOURS["MLP"]),
    ("Move object", "CNN", "same object\nstays stable", MODEL_COLOURS["CNN"]),
    ("Morph shape", "MLP", "score depends\non position", MODEL_COLOURS["MLP"]),
    ("Morph shape", "CNN", "score follows\nshape", MODEL_COLOURS["CNN"]),
    ("Hide evidence", "MLP", f"wrong score\ndrops {best_patch_drop * 100:.1f}pp", MODEL_COLOURS["MLP"]),
    ("Feature space", "CNN", "neighbours/probes\nmore shape-stable", MODEL_COLOURS["CNN"]),
]
for idx, (probe, model, result, colour) in enumerate(summary_cells):
    row = idx // 2
    col = idx % 2
    x0 = 0.08 + col * 0.46
    y0 = 0.70 - row * 0.28
    axis.add_patch(plt.Rectangle((x0, y0), 0.38, 0.20, facecolor="#F7F7F7", edgecolor=colour, linewidth=2.0, transform=axis.transAxes))
    axis.text(x0 + 0.03, y0 + 0.145, probe, fontsize=10.5, color="#555555", transform=axis.transAxes)
    axis.text(x0 + 0.03, y0 + 0.085, model, fontsize=14, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(x0 + 0.20, y0 + 0.074, result, fontsize=12, fontweight="bold", ha="center", va="center", transform=axis.transAxes)
save_and_show(fig, "fig16_what_changes_the_decision.png")


## 11. Intervention and re-test

The intervention is a better modelling and data strategy for this controlled task: a translation-aware CNN trained with position augmentation. The point is not that CNNs solve all shortcuts. The point is that we changed the learning incentives, then re-tested the same shortcut-breaking cases.


### 11.1 Claim-evidence-risk board

The final ledger is the case file: no single heatmap proves the shortcut, but the behavioural, response-map, morph, saliency, and re-test evidence point in the same direction.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/06_shortcut_evidence_ledger.png" style="width:100%; border:1px solid #ddd">


In [ ]:
ledger_sections = [
    (
        "Behavioural evidence",
        [
            "IID validation hides the shortcut.",
            "OOD position audit exposes the MLP failure.",
            "Same-shape movement flips the MLP but not the CNN.",
        ],
        MODEL_COLOURS["MLP"],
    ),
    (
        "Model interrogation",
        [
            "Position-response maps show the MLP score follows location.",
            "Shape-position surfaces show MLP score is position-dominated.",
            "Shape morphs and movement paths ask what changes the score.",
        ],
        MODEL_COLOURS["CNN"],
    ),
    (
        "Explanation caveat",
        [
            "Saliency alone is not enough.",
            "Area-normalised relevance supports, but does not prove, the diagnosis.",
            "Average relevance maps are global diagnostics, not causal evidence.",
        ],
        "#777777",
    ),
    (
        "Representation evidence",
        [
            "Nearest neighbours show what each feature space treats as similar.",
            "Linear probes test whether shape or position is easier to decode.",
            "These are intrinsic diagnostics, not deployment guarantees.",
        ],
        "#6A5ACD",
    ),
    (
        "Evidence removal",
        [
            f"Top-ranked masking weakens the wrong MLP score by {best_patch_drop * 100:.1f}pp.",
            "Low-importance masking is the control path.",
            "Movement remains the cleaner counterfactual for position shortcuts.",
        ],
        "#B56576",
    ),
    (
        "Re-test",
        [
            "Shared convolution, global pooling, and position augmentation reduce the shortcut.",
            f"MLP crossed accuracy {mlp_crossed_accuracy * 100:.1f}%; CNN crossed accuracy {cnn_crossed_accuracy * 100:.1f}%.",
            "The CNN is more translation-aware here, not certified robust in general.",
        ],
        "#444444",
    ),
]

fig, axes = plt.subplots(2, 3, figsize=(14.6, 7.6), constrained_layout=True)
fig.suptitle("Shortcut evidence ledger", fontsize=18, fontweight="bold")
for axis, (title, bullets, colour) in zip(axes.flat, ledger_sections, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.02, 0.05), 0.96, 0.88, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.0, transform=axis.transAxes))
    axis.text(0.07, 0.82, title, fontsize=13.5, fontweight="bold", color=colour, transform=axis.transAxes)
    for index, bullet in enumerate(bullets):
        axis.text(0.09, 0.63 - index * 0.17, f"• {bullet}", fontsize=10.2, transform=axis.transAxes, wrap=True)
fig.text(
    0.5,
    -0.02,
    "No single heatmap proves the shortcut. The diagnosis is compelling because behaviour, counterfactual movement, response maps, morphs, attribution, and representation checks point to the same position dependence.",
    ha="center",
    fontsize=11.5,
    fontweight="bold",
)
save_and_show(fig, "fig17_evidence_ledger.png")


## 11.2 Behavioural evidence in motion

The exported files are useful for slides. The sequence is what matters: **move the object, watch the confidence path, inspect the changing heatmaps, and re-test after changing the learning incentives**.

The heatmap overlays are deliberately secondary. They show how the saliency picture changes frame by frame during movement and morphing, but the strongest evidence remains behavioural: only position changes, and the score changes with it.


In [ ]:
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import HTML, Image as IPyImage, Video

PRESENTATION_FILES = [
    "fig00_apparent_shape_task.png",
    "fig00_model_cards.png",
    "fig01_both_models_appear_to_work.png",
    "fig_final_bridge_to_real_xai.png",
    "fig_final_xai_loop.png",
    "fig_final_real_world_bridge.png",
    "00_hidden_position_shortcut.png",
    "01_same_shape_movement_counterfactual.png",
    "02_movement_confidence_paths.png",
    "03_position_response_maps_with_boundaries.png",
    "04_shape_position_score_surface.png",
    "05_why_heatmaps_are_not_enough.png",
    "06_shortcut_evidence_ledger.png",
    "anim_moon_moves_confidence.gif",
    "anim_star_moves_confidence.gif",
    "anim_moon_moves_confidence.mp4",
    "anim_star_moves_confidence.mp4",
    "anim_response_map_path_mlp.gif",
    "anim_response_map_path_cnn.gif",
    "anim_morph_lower_left.gif",
    "anim_morph_upper_right.gif",
    "anim_moon_moves_heatmaps.gif",
    "anim_moon_moves_heatmaps.mp4",
    "anim_star_moves_heatmaps.gif",
    "anim_star_moves_heatmaps.mp4",
    "anim_morph_lower_left_heatmaps.gif",
    "anim_morph_lower_left_heatmaps.mp4",
    "anim_morph_upper_right_heatmaps.gif",
    "anim_morph_upper_right_heatmaps.mp4",
    "fig20_act2_apparent_shape_task_invisible_background.png",
    "fig20a_background_only_sanity_check.png",
    "fig21_act2_cnn_appears_to_work.png",
    "fig22_invisible_background_swap_counterfactual.png",
    "fig23_invisible_background_confidence_sweep.png",
    "fig24_invisible_background_difference_amplified.png",
    "fig25_background_tint_response_surface.png",
    "fig26_background_vs_shape_bars.png",
    "fig27_act2_heatmaps_are_not_enough.png",
    "fig28_act2_mitigation_retest.png",
    "fig29_two_act_evidence_board.png",
    "anim_invisible_background_morph_moon.gif",
    "anim_invisible_background_morph_moon.mp4",
    "anim_invisible_background_morph_star.gif",
    "anim_invisible_background_morph_star.mp4",
]
presentation_export_manifest: dict[str, str] = {}


def export_presentation_figure(fig: plt.Figure, filename: str) -> Path:
    path = FIGURE_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white", dpi=190)
    presentation_export_manifest[filename] = str(path.relative_to(PROJECT_ROOT))
    plt.show()
    plt.close(fig)
    return path


def save_gif(frames: list[Image.Image], filename: str, duration_ms: int = 130) -> Path:
    path = FIGURE_DIR / filename
    rgb_frames = [frame.convert("RGB") for frame in frames]
    rgb_frames[0].save(path, save_all=True, append_images=rgb_frames[1:], duration=duration_ms, loop=0)
    presentation_export_manifest[filename] = str(path.relative_to(PROJECT_ROOT))
    display(IPyImage(filename=str(path)))
    return path


def save_mp4_if_possible(frames: list[Image.Image], filename: str, fps: int = 8) -> None:
    path = FIGURE_DIR / filename
    try:
        import imageio.v3 as iio  # type: ignore[import-not-found]
    except Exception as error:
        print(f"MP4 export skipped for {filename}: imageio is unavailable ({error}). GIF export was created.")
        return
    try:
        arrays = [np.asarray(frame.convert("RGB")) for frame in frames]
        iio.imwrite(path, arrays, fps=fps, macro_block_size=1)
        presentation_export_manifest[filename] = str(path.relative_to(PROJECT_ROOT))
    except Exception as error:
        print(f"MP4 export skipped for {filename}: {error}. GIF export was created.")





def save_pil_presentation_image(image: Image.Image, filename: str) -> Path:
    path = FIGURE_DIR / filename
    image.convert("RGB").save(path)
    presentation_export_manifest[filename] = str(path.relative_to(PROJECT_ROOT))
    display(IPyImage(filename=str(path)))
    return path


def pil_font(size: int, *, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Helvetica Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Helvetica.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size)
        except OSError:
            continue
    return ImageFont.load_default()


def draw_pil_confidence_bar(
    draw: ImageDraw.ImageDraw,
    xy: tuple[int, int],
    width: int,
    score: float,
    colour: tuple[int, int, int],
    label: str,
    *,
    font: ImageFont.ImageFont,
) -> None:
    x, y = xy
    draw.rounded_rectangle((x, y, x + width, y + 34), radius=9, fill=(238, 238, 238), outline=(195, 195, 195), width=2)
    draw.rounded_rectangle((x, y, x + int(width * score), y + 34), radius=9, fill=colour)
    draw.text((x, y - 30), label, fill=(30, 30, 30), font=font)


def save_gif_or_warn(frames: list[Image.Image], gif_filename: str, mp4_filename: str | None = None, *, fps: int = 8, duration_ms: int = 130) -> None:
    save_gif(frames, gif_filename, duration_ms=duration_ms)
    if mp4_filename is not None:
        save_mp4_if_possible(frames, mp4_filename, fps=fps)


def register_existing_presentation_file(filename: str) -> None:
    path = FIGURE_DIR / filename
    if path.exists():
        presentation_export_manifest[filename] = str(path.relative_to(PROJECT_ROOT))


for existing_filename in [
    "fig00_apparent_shape_task.png",
    "fig00_model_cards.png",
    "fig01_both_models_appear_to_work.png",
    "fig01_shape_position_grid.png",
    "fig02_training_position_distribution.png",
    "fig05_same_shape_moved.png",
    "fig06_position_response_maps.png",
    "fig07_position_decision_boundary.png",
    "fig08_shape_morph_strip.png",
    "fig09_shape_position_surface.png",
    "fig10_movement_path.png",
    "fig11_saliency_comparison.png",
    "fig15_minimal_evidence_removal.png",
    "fig16_what_changes_the_decision.png",
    "fig17_evidence_ledger.png",
]:
    register_existing_presentation_file(existing_filename)



def draw_axis_confidence_bar(axis: plt.Axes, x0: float, y0: float, width: float, score: float, colour: str) -> None:
    axis.add_patch(plt.Rectangle((x0, y0), width, 0.035, transform=axis.transAxes, facecolor="#EFEFEF", edgecolor="#CCCCCC", linewidth=0.8, clip_on=False))
    axis.add_patch(plt.Rectangle((x0, y0), width * float(score), 0.035, transform=axis.transAxes, facecolor=colour, edgecolor="none", clip_on=False))


def star_probability_label(score: float) -> str:
    return f"star {score:.2f}" if score >= 0.5 else f"moon {1.0 - score:.2f}"


def draw_score_bar(draw: ImageDraw.ImageDraw, xy: tuple[int, int], width: int, score: float, colour: str, label: str) -> None:
    x, y = xy
    draw.rounded_rectangle((x, y, x + width, y + 26), radius=8, fill=(238, 238, 238), outline=(190, 190, 190))
    draw.rounded_rectangle((x, y, x + int(width * score), y + 26), radius=8, fill=colour)
    draw.text((x, y - 20), f"{label} P(star) {score:.2f} -> {star_probability_label(score)}", fill=(30, 30, 30))


def confidence_frame(
    sample: PositionSample,
    mlp_score: float,
    cnn_score: float,
    progress: float,
    title: str,
    mlp_crossing: float | None,
) -> Image.Image:
    frame = Image.new("RGB", (840, 480), (250, 250, 250))
    draw = ImageDraw.Draw(frame)
    draw.text((28, 22), title, fill=(25, 25, 25))
    draw.text((28, 52), f"movement progress: {progress:.2f}", fill=(80, 80, 80))
    canvas = sample.image.resize((300, 300), Image.Resampling.LANCZOS).convert("RGB")
    frame.paste(canvas, (38, 105))
    draw_score_bar(draw, (390, 150), 360, mlp_score, (196, 78, 82), "MLP")
    draw_score_bar(draw, (390, 235), 360, cnn_score, (46, 139, 87), "CNN")
    draw.line((390, 333, 750, 333), fill=(35, 35, 35), width=3)
    draw.ellipse((390 + int(360 * progress) - 7, 326, 390 + int(360 * progress) + 7, 340), fill=(20, 20, 20))
    draw.text((390, 350), "lower-left", fill=(75, 75, 75))
    draw.text((675, 350), "upper-right", fill=(75, 75, 75))
    if mlp_crossing is not None and progress >= mlp_crossing:
        draw.rounded_rectangle((390, 385, 750, 425), radius=10, fill=(255, 232, 232), outline=(196, 78, 82), width=2)
        draw.text((408, 397), f"MLP has crossed its decision boundary at {mlp_crossing:.2f}", fill=(140, 40, 45))
    return frame


def figure_to_image(fig: plt.Figure) -> Image.Image:
    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    array = np.asarray(canvas.buffer_rgba())
    image = Image.fromarray(array).convert("RGB")
    plt.close(fig)
    return image


# A. Hidden exam leak / position shortcut figure.
fig, axis = plt.subplots(figsize=(8.6, 7.8), constrained_layout=True)
fig.suptitle("The hidden shortcut: position predicts the label", fontsize=19, fontweight="bold")
for region, label_text, colour in [
    (LOWER_LEFT_REGION, "moon territory", CLASS_COLOURS[0]),
    (UPPER_RIGHT_REGION, "star territory", CLASS_COLOURS[1]),
]:
    x0, x1, y0, y1 = region
    axis.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=colour, edgecolor=colour, alpha=0.16, linewidth=2.8))
    axis.text((x0 + x1) / 2, (y0 + y1) / 2, label_text, ha="center", va="center", color=colour, fontsize=14, fontweight="bold")
for label, marker, size in [(0, "o", 46), (1, "*", 82)]:
    pts = centres_for(biased_train_samples, label)
    axis.scatter(pts[:, 0], pts[:, 1], s=size, marker=marker, color=CLASS_COLOURS[label], alpha=0.82, edgecolor="white", linewidth=0.55, label=f"training {CLASS_NAMES[label]}")
axis.set_xlim(0, IMAGE_SIZE)
axis.set_ylim(IMAGE_SIZE, 0)
axis.set_aspect("equal")
axis.set_xlabel("object centre x")
axis.set_ylabel("object centre y")
axis.legend(frameon=False, loc="lower right")
axis.text(0.5, -0.09, "The hidden exam leak: location is predictive even though the true task is shape.", transform=axis.transAxes, ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "00_hidden_position_shortcut.png")

# B. Same-shape counterfactual panel.
def draw_counterfactual_card(
    canvas: Image.Image,
    draw: ImageDraw.ImageDraw,
    box: tuple[int, int, int, int],
    title: str,
    sample: PositionSample,
    mlp_score: float,
    cnn_score: float,
) -> None:
    x0, y0, x1, y1 = box
    draw.rounded_rectangle((x0, y0, x1, y1), radius=22, fill=(250, 250, 250), outline=(220, 220, 220), width=2)
    draw.text((x0 + 24, y0 + 18), title, fill=(20, 20, 20), font=font_subtitle)
    draw.text((x0 + 24, y0 + 58), f"TRUE: {CLASS_NAMES[sample.label]} | POSITION: {sample.position.replace('_', '-')}", fill=(70, 70, 70), font=font_body)
    canvas.paste(sample.image.resize((330, 330), Image.Resampling.LANCZOS).convert("RGB"), (x0 + 32, y0 + 96))
    for row, model_name, score in [(0, "MLP", mlp_score), (1, "CNN", cnn_score)]:
        pred = int(score >= 0.5)
        ok = pred == sample.label
        colour_hex = MODEL_COLOURS[model_name] if ok else "#B23A48"
        colour = tuple(int(colour_hex.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4))
        y = y0 + 152 + row * 118
        draw.text((x0 + 405, y - 42), f"{model_name}: {CLASS_NAMES[pred]} | P(star)={score:.2f}", fill=colour, font=font_body_bold)
        draw_pil_confidence_bar(draw, (x0 + 405, y), 420, float(score), colour, "intended evidence" if ok else "shortcut exposed", font=font_small_bold)

font_title = pil_font(48, bold=True)
font_subtitle = pil_font(28, bold=True)
font_body = pil_font(22)
font_body_bold = pil_font(24, bold=True)
font_small_bold = pil_font(18, bold=True)
canvas = Image.new("RGB", (1900, 1420), (255, 255, 255))
draw = ImageDraw.Draw(canvas)
draw.text((95, 42), "Same moon. Same pixels. Different place. Different answer.", fill=(10, 10, 10), font=font_title)
cards = [
    ((70, 130, 920, 625), "Same moon: lower-left", moon_lower_left, float(cf_scores["MLP"][0]), float(cf_scores["CNN"][0])),
    ((980, 130, 1830, 625), "Same moon: upper-right", moon_upper_right, float(cf_scores["MLP"][1]), float(cf_scores["CNN"][1])),
    ((70, 690, 920, 1185), "Same star: upper-right", star_upper_right, float(cf_scores["MLP"][2]), float(cf_scores["CNN"][2])),
    ((980, 690, 1830, 1185), "Same star: lower-left", star_lower_left, float(cf_scores["MLP"][3]), float(cf_scores["CNN"][3])),
]
for card in cards:
    draw_counterfactual_card(canvas, draw, *card)
draw.rounded_rectangle((185, 1235, 1715, 1302), radius=18, fill=(244, 244, 244), outline=(210, 210, 210), width=2)
draw.text((240, 1254), "Changed: position.  Unchanged: shape seed, rendered object, and label meaning.", fill=(20, 20, 20), font=font_body_bold)
draw.text((285, 1342), "The MLP prediction changes when the address changes. The CNN is more shape-stable on the same cases.", fill=(20, 20, 20), font=font_body_bold)
save_pil_presentation_image(canvas, "01_same_shape_movement_counterfactual.png")

# C and D. Movement animations and static confidence path.
moon_crossing = first_crossing(movement_progress, movement_path_results["moon_mlp"], starts_above=False)
star_crossing = first_crossing(movement_progress, movement_path_results["star_mlp"], starts_above=True)
moon_frames = [
    confidence_frame(sample, float(movement_path_results["moon_mlp"][index]), float(movement_path_results["moon_cnn"][index]), float(movement_progress[index]), "Same moon moves: lower-left -> upper-right", moon_crossing)
    for index, sample in enumerate(moon_movement_samples)
]
star_frames = [
    confidence_frame(sample, float(movement_path_results["star_mlp"][index]), float(movement_path_results["star_cnn"][index]), float(movement_progress[index]), "Same star moves: upper-right -> lower-left", star_crossing)
    for index, sample in enumerate(star_movement_samples)
]
save_gif(moon_frames, "anim_moon_moves_confidence.gif")
save_gif(star_frames, "anim_star_moves_confidence.gif")
save_mp4_if_possible(moon_frames, "anim_moon_moves_confidence.mp4")
save_mp4_if_possible(star_frames, "anim_star_moves_confidence.mp4")

fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2), constrained_layout=True)
fig.suptitle("How far must the object move before the model changes its mind?", fontsize=18, fontweight="bold")
for axis, title, mlp_key, cnn_key, crossing in [
    (axes[0], "Moon path: lower-left -> upper-right", "moon_mlp", "moon_cnn", moon_crossing),
    (axes[1], "Star path: upper-right -> lower-left", "star_mlp", "star_cnn", star_crossing),
]:
    axis.plot(movement_progress, movement_path_results[mlp_key], color=MODEL_COLOURS["MLP"], linewidth=2.8, label="MLP")
    axis.plot(movement_progress, movement_path_results[cnn_key], color=MODEL_COLOURS["CNN"], linewidth=2.8, label="CNN")
    axis.axhline(0.5, color="black", linestyle="--", linewidth=1.1, label="decision threshold")
    if crossing is not None:
        axis.axvline(crossing, color=MODEL_COLOURS["MLP"], linestyle=":", linewidth=2.2)
        axis.text(crossing + 0.02, 0.07, f"MLP flips at {crossing:.2f}", color=MODEL_COLOURS["MLP"], fontweight="bold")
    axis.set_title(title)
    axis.set_xlabel("movement progress")
    axis.set_ylabel("P(star)")
    axis.set_ylim(-0.03, 1.03)
    axis.grid(alpha=0.18)
    axis.legend(frameon=False, loc="best")
fig.text(0.5, -0.025, "Movement alone changes the MLP decision. That is the behavioural counterfactual evidence.", ha="center", fontsize=12.2, fontweight="bold")
export_presentation_figure(fig, "02_movement_confidence_paths.png")

# E. Position-response maps with contours and boundaries.
fig, axes = plt.subplots(2, 2, figsize=(12.4, 9.4), constrained_layout=True)
fig.suptitle("Where does the model believe stars live?", fontsize=19, fontweight="bold")
response_upgrade_panels = [
    (axes[0, 0], "MLP fixed moon", moon_position_response_mlp, moon_lower_left.centre, moon_upper_right.centre),
    (axes[0, 1], "CNN fixed moon", moon_position_response_cnn, moon_lower_left.centre, moon_upper_right.centre),
    (axes[1, 0], "MLP fixed star", star_position_response_mlp, star_upper_right.centre, star_lower_left.centre),
    (axes[1, 1], "CNN fixed star", star_position_response_cnn, star_upper_right.centre, star_lower_left.centre),
]
for axis, title, matrix, start, end in response_upgrade_panels:
    im = axis.imshow(matrix, cmap="RdYlBu_r", vmin=0, vmax=1, extent=[SCAN_VALUES[0], SCAN_VALUES[-1], SCAN_VALUES[-1], SCAN_VALUES[0]], interpolation="bicubic")
    contour_set = axis.contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.25, 0.5, 0.75], colors=["#666666", "black", "#666666"], linewidths=[1.0, 2.1, 1.0])
    axis.clabel(contour_set, fmt={0.25: "0.25", 0.5: "decision boundary", 0.75: "0.75"}, fontsize=8)
    for region, colour in [(LOWER_LEFT_REGION, CLASS_COLOURS[0]), (UPPER_RIGHT_REGION, CLASS_COLOURS[1])]:
        axis.add_patch(plt.Rectangle((region[0], region[2]), region[1] - region[0], region[3] - region[2], fill=False, edgecolor=colour, linewidth=2.2))
    axis.annotate("", xy=end, xytext=start, arrowprops={"arrowstyle": "->", "color": "white", "lw": 2.4, "linestyle": "--"})
    axis.scatter([start[0]], [start[1]], s=70, color="white", edgecolor="#2E8B57", linewidth=2.0, zorder=5)
    axis.scatter([end[0]], [end[1]], s=70, color="white", edgecolor="#B23A48", linewidth=2.0, zorder=5)
    axis.set_title(title)
    axis.set_xlabel("object centre x")
    axis.set_ylabel("object centre y")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, label="P(star)")
fig.text(0.5, -0.02, "The MLP has a territory map. The same moon crosses into star territory when moved upper-right.", ha="center", fontsize=12.2, fontweight="bold")
export_presentation_figure(fig, "03_position_response_maps_with_boundaries.png")

# F. Response-map path animations.
def response_map_path_frame(matrix: np.ndarray, sample: PositionSample, centre: tuple[int, int], progress: float, title: str) -> Image.Image:
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.4), constrained_layout=True)
    fig.suptitle(title, fontsize=15, fontweight="bold")
    axes[0].imshow(matrix, cmap="RdYlBu_r", vmin=0, vmax=1, extent=[SCAN_VALUES[0], SCAN_VALUES[-1], SCAN_VALUES[-1], SCAN_VALUES[0]], interpolation="bicubic")
    axes[0].contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.5], colors="black", linewidths=1.8)
    axes[0].scatter([centre[0]], [centre[1]], s=95, color="white", edgecolor="black", linewidth=2.0, zorder=6)
    axes[0].set_title("position-response map")
    axes[0].set_xlim(SCAN_VALUES[0], SCAN_VALUES[-1])
    axes[0].set_ylim(SCAN_VALUES[-1], SCAN_VALUES[0])
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[1].imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"same moon, progress {progress:.2f}")
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    return figure_to_image(fig)

mlp_response_frames = [
    response_map_path_frame(moon_position_response_mlp, sample, sample.centre, float(movement_progress[index]), "MLP: same moon crossing the territory map")
    for index, sample in enumerate(moon_movement_samples)
]
cnn_response_frames = [
    response_map_path_frame(moon_position_response_cnn, sample, sample.centre, float(movement_progress[index]), "CNN: same moon remains shape-stable")
    for index, sample in enumerate(moon_movement_samples)
]
save_gif(mlp_response_frames, "anim_response_map_path_mlp.gif", duration_ms=120)
save_gif(cnn_response_frames, "anim_response_map_path_cnn.gif", duration_ms=120)

# G. Shape morph animations.
def morph_frame(alpha: float, centre: tuple[int, int], title: str) -> Image.Image:
    image = render_morph_image(alpha, centre, SEED + 93_000)
    mlp_score = score_image(baseline_mlp, image)
    cnn_score = score_image(mitigated_cnn, image)
    address = "lower-left" if centre == CANONICAL_CENTRES["lower_left"] else "upper-right"
    frame = Image.new("RGB", (840, 480), (250, 250, 250))
    draw = ImageDraw.Draw(frame)
    draw.text((28, 22), title, fill=(25, 25, 25))
    draw.text((28, 52), f"fixed address: {address} | morph alpha: {alpha:.2f}", fill=(80, 80, 80))
    frame.paste(image.resize((300, 300), Image.Resampling.LANCZOS).convert("RGB"), (38, 105))
    draw_score_bar(draw, (390, 150), 360, mlp_score, (196, 78, 82), "MLP")
    draw_score_bar(draw, (390, 235), 360, cnn_score, (46, 139, 87), "CNN")
    draw.line((390, 333, 750, 333), fill=(35, 35, 35), width=3)
    draw.ellipse((390 + int(360 * alpha) - 7, 326, 390 + int(360 * alpha) + 7, 340), fill=(20, 20, 20))
    draw.text((390, 350), "moon shape", fill=(75, 75, 75))
    draw.text((675, 350), "star shape", fill=(75, 75, 75))
    if alpha >= 0.75 and cnn_score >= 0.5:
        draw.rounded_rectangle((390, 385, 750, 425), radius=10, fill=(232, 247, 238), outline=(46, 139, 87), width=2)
        draw.text((408, 397), "CNN has followed the shape morph", fill=(35, 105, 65))
    return frame

morph_animation_alphas = np.linspace(0, 1, 21)
lower_left_morph_frames = [morph_frame(float(alpha), CANONICAL_CENTRES["lower_left"], "Morph at lower-left: moon -> star") for alpha in morph_animation_alphas]
upper_right_morph_frames = [morph_frame(float(alpha), CANONICAL_CENTRES["upper_right"], "Morph at upper-right: moon -> star") for alpha in morph_animation_alphas]
save_gif(lower_left_morph_frames, "anim_morph_lower_left.gif", duration_ms=105)
save_gif(upper_right_morph_frames, "anim_morph_upper_right.gif", duration_ms=105)



# G2. Animated heatmap overlays: keep saliency visual and secondary.
def heatmap_animation_frame(
    sample: PositionSample,
    index: int,
    mlp_key: str,
    cnn_key: str,
    title: str,
    progress_label: str,
) -> Image.Image:
    mlp_score = float(movement_path_results[mlp_key][index])
    cnn_score = float(movement_path_results[cnn_key][index])
    mlp_saliency = smoothgrad_saliency(baseline_mlp, sample.image, 1, samples=1, noise=0.025)
    cnn_saliency = smoothgrad_saliency(mitigated_cnn, sample.image, 1, samples=1, noise=0.025)

    fig = plt.figure(figsize=(12.8, 6.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 4, width_ratios=[1.0, 1.0, 1.0, 1.45], height_ratios=[1.0, 0.72])
    fig.suptitle(title, fontsize=17, fontweight="bold")

    for axis, panel_title, image, saliency in [
        (fig.add_subplot(grid[0, 0]), "same object", sample.image, None),
        (fig.add_subplot(grid[0, 1]), "MLP SmoothGrad", sample.image, mlp_saliency),
        (fig.add_subplot(grid[0, 2]), "CNN SmoothGrad", sample.image, cnn_saliency),
    ]:
        axis.imshow(image.resize((MODEL_SIZE, MODEL_SIZE)), cmap="gray", vmin=0, vmax=255)
        if saliency is not None:
            axis.imshow(saliency, cmap="magma", alpha=0.68)
        axis.set_title(panel_title)
        axis.set_xticks([])
        axis.set_yticks([])

    score_axis = fig.add_subplot(grid[0, 3])
    bars = score_axis.barh(
        ["MLP", "CNN"],
        [mlp_score, cnn_score],
        color=[MODEL_COLOURS["MLP"], MODEL_COLOURS["CNN"]],
        alpha=0.9,
    )
    score_axis.axvline(0.5, color="black", linestyle="--", linewidth=1.1)
    score_axis.set_xlim(0, 1)
    score_axis.set_xlabel("P(star)")
    score_axis.set_title(progress_label)
    for bar, score in zip(bars, [mlp_score, cnn_score], strict=True):
        score_axis.text(score + 0.025, bar.get_y() + bar.get_height() / 2, f"{score:.2f}", va="center", fontsize=10.5, fontweight="bold")

    path_axis = fig.add_subplot(grid[1, :])
    path_axis.plot(movement_progress, movement_path_results[mlp_key], color=MODEL_COLOURS["MLP"], linewidth=2.6, label="MLP P(star)")
    path_axis.plot(movement_progress, movement_path_results[cnn_key], color=MODEL_COLOURS["CNN"], linewidth=2.6, label="CNN P(star)")
    path_axis.scatter([movement_progress[index]], [mlp_score], color=MODEL_COLOURS["MLP"], s=80, zorder=5)
    path_axis.scatter([movement_progress[index]], [cnn_score], color=MODEL_COLOURS["CNN"], s=80, zorder=5)
    path_axis.axhline(0.5, color="black", linestyle="--", linewidth=1.0)
    path_axis.set_ylim(-0.03, 1.03)
    path_axis.set_xlabel("movement progress")
    path_axis.set_ylabel("P(star)")
    path_axis.grid(alpha=0.16)
    path_axis.legend(frameon=False, loc="best")
    fig.text(0.5, -0.02, "The heatmap changes with the frame, but the behavioural evidence is the score changing when only position changes.", ha="center", fontsize=11.2, fontweight="bold")
    return figure_to_image(fig)


heatmap_frame_indices = np.linspace(0, len(moon_movement_samples) - 1, 7, dtype=int)
torch.manual_seed(SEED + 8_801)
moon_heatmap_frames = [
    heatmap_animation_frame(
        moon_movement_samples[int(index)],
        int(index),
        "moon_mlp",
        "moon_cnn",
        "Same moon moves: heatmaps update while the score changes",
        f"progress {movement_progress[int(index)]:.2f}",
    )
    for index in heatmap_frame_indices
]
torch.manual_seed(SEED + 8_802)
star_heatmap_frames = [
    heatmap_animation_frame(
        star_movement_samples[int(index)],
        int(index),
        "star_mlp",
        "star_cnn",
        "Same star moves: heatmaps update while the score changes",
        f"progress {movement_progress[int(index)]:.2f}",
    )
    for index in heatmap_frame_indices
]
save_gif(moon_heatmap_frames, "anim_moon_moves_heatmaps.gif", duration_ms=150)
save_gif(star_heatmap_frames, "anim_star_moves_heatmaps.gif", duration_ms=150)
save_mp4_if_possible(moon_heatmap_frames, "anim_moon_moves_heatmaps.mp4", fps=7)
save_mp4_if_possible(star_heatmap_frames, "anim_star_moves_heatmaps.mp4", fps=7)


def morph_heatmap_frame(alpha: float, centre: tuple[int, int], title: str) -> Image.Image:
    image = render_morph_image(alpha, centre, SEED + 93_000)
    mlp_score = score_image(baseline_mlp, image)
    cnn_score = score_image(mitigated_cnn, image)
    mlp_saliency = smoothgrad_saliency(baseline_mlp, image, 1, samples=1, noise=0.025)
    cnn_saliency = smoothgrad_saliency(mitigated_cnn, image, 1, samples=1, noise=0.025)

    fig = plt.figure(figsize=(11.8, 5.8), constrained_layout=True)
    grid = fig.add_gridspec(1, 4, width_ratios=[1.0, 1.0, 1.0, 1.25])
    fig.suptitle(title, fontsize=17, fontweight="bold")
    for axis, panel_title, saliency in [
        (fig.add_subplot(grid[0, 0]), "morphed shape", None),
        (fig.add_subplot(grid[0, 1]), "MLP SmoothGrad", mlp_saliency),
        (fig.add_subplot(grid[0, 2]), "CNN SmoothGrad", cnn_saliency),
    ]:
        axis.imshow(image.resize((MODEL_SIZE, MODEL_SIZE)), cmap="gray", vmin=0, vmax=255)
        if saliency is not None:
            axis.imshow(saliency, cmap="magma", alpha=0.68)
        axis.set_title(panel_title)
        axis.set_xticks([])
        axis.set_yticks([])

    score_axis = fig.add_subplot(grid[0, 3])
    bars = score_axis.barh(
        ["MLP", "CNN"],
        [mlp_score, cnn_score],
        color=[MODEL_COLOURS["MLP"], MODEL_COLOURS["CNN"]],
        alpha=0.9,
    )
    score_axis.axvline(0.5, color="black", linestyle="--", linewidth=1.1)
    score_axis.set_xlim(0, 1)
    score_axis.set_xlabel("P(star)")
    score_axis.set_title(f"morph alpha {alpha:.2f}")
    for bar, score in zip(bars, [mlp_score, cnn_score], strict=True):
        score_axis.text(score + 0.025, bar.get_y() + bar.get_height() / 2, f"{score:.2f}", va="center", fontsize=10.5, fontweight="bold")
    fig.text(0.5, -0.02, "Morph heatmaps are useful context; the key question is whether the score follows shape consistently at a fixed address.", ha="center", fontsize=11.2, fontweight="bold")
    return figure_to_image(fig)


morph_heatmap_alphas = np.linspace(0, 1, 7)
torch.manual_seed(SEED + 8_803)
lower_left_morph_heatmap_frames = [
    morph_heatmap_frame(float(alpha), CANONICAL_CENTRES["lower_left"], "Moon-to-star morph at lower-left: heatmaps and scores")
    for alpha in morph_heatmap_alphas
]
torch.manual_seed(SEED + 8_804)
upper_right_morph_heatmap_frames = [
    morph_heatmap_frame(float(alpha), CANONICAL_CENTRES["upper_right"], "Moon-to-star morph at upper-right: heatmaps and scores")
    for alpha in morph_heatmap_alphas
]
save_gif(lower_left_morph_heatmap_frames, "anim_morph_lower_left_heatmaps.gif", duration_ms=150)
save_gif(upper_right_morph_heatmap_frames, "anim_morph_upper_right_heatmaps.gif", duration_ms=150)
save_mp4_if_possible(lower_left_morph_heatmap_frames, "anim_morph_lower_left_heatmaps.mp4", fps=7)
save_mp4_if_possible(upper_right_morph_heatmap_frames, "anim_morph_upper_right_heatmaps.mp4", fps=7)

# H. Shape-position surface export under presentation filename.
fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.8), constrained_layout=True)
fig.suptitle("Shape versus position: what controls the score?", fontsize=19, fontweight="bold")
for axis, title, surface, note in [
    (axes[0], "MLP", shape_position_surface_mlp, "position dominates"),
    (axes[1], "CNN", shape_position_surface_cnn, "shape dominates"),
]:
    im = axis.imshow(surface, cmap="RdYlBu_r", vmin=0, vmax=1, origin="lower", aspect="auto", extent=[0, 1, 0, 1], interpolation="bicubic")
    axis.contour(position_alphas, morph_alphas, surface, levels=[0.5], colors="black", linewidths=1.7)
    axis.text(0.05, 0.88, note, transform=axis.transAxes, fontsize=12.5, color="white", fontweight="bold", bbox={"boxstyle": "round,pad=0.35", "facecolor": "black", "alpha": 0.72, "edgecolor": "none"})
    axis.set_title(title)
    axis.set_xlabel("morph alpha: moon -> star")
    axis.set_ylabel("position path: lower-left -> upper-right")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.86, label="P(star)")
fig.text(0.5, -0.025, "If the surface changes mainly left-to-right, shape controls the score. If it changes mainly bottom-to-top, position controls it.", ha="center", fontsize=11.8)
export_presentation_figure(fig, "04_shape_position_score_surface.png")

# I. Saliency caution figure.
fig = plt.figure(figsize=(14.4, 6.6), constrained_layout=True)
grid = fig.add_gridspec(2, 4, width_ratios=[1.0, 1.0, 1.05, 1.25])
fig.suptitle("Why heatmaps are not enough", fontsize=19, fontweight="bold")
for row, sample, saliency, response_matrix, path_key, title in [
    (0, moved_moon_failure, saliency_maps[0][2], moon_position_response_mlp, "moon_mlp", "MLP moved moon failure"),
    (1, moved_star_failure, saliency_maps[2][2], star_position_response_mlp, "star_mlp", "MLP moved star failure"),
]:
    axes = [fig.add_subplot(grid[row, col]) for col in range(4)]
    axes[0].imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title(f"original\n{title}", fontsize=10)
    axes[1].imshow(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), cmap="gray", alpha=0.55)
    axes[1].imshow(saliency, cmap="magma", alpha=0.66)
    axes[1].set_title("saliency", fontsize=10)
    axes[2].imshow(response_matrix, cmap="RdYlBu_r", vmin=0, vmax=1, extent=[SCAN_VALUES[0], SCAN_VALUES[-1], SCAN_VALUES[-1], SCAN_VALUES[0]], interpolation="bicubic")
    axes[2].contour(SCAN_VALUES, SCAN_VALUES, response_matrix, levels=[0.5], colors="black", linewidths=1.6)
    axes[2].scatter([sample.centre[0]], [sample.centre[1]], color="white", edgecolor="black", s=60, zorder=5)
    axes[2].set_title("position-response map", fontsize=10)
    axes[3].plot(movement_progress, movement_path_results[path_key], color=MODEL_COLOURS["MLP"], linewidth=2.5)
    axes[3].axhline(0.5, color="black", linestyle="--", linewidth=1.0)
    axes[3].set_ylim(-0.03, 1.03)
    axes[3].set_title("movement confidence path", fontsize=10)
    axes[3].set_xlabel("movement progress")
    axes[3].set_ylabel("P(star)")
    axes[3].grid(alpha=0.16)
    for axis in axes[:3]:
        axis.set_xticks([])
        axis.set_yticks([])
fig.text(0.5, -0.02, "Saliency may highlight the object, but behavioural probes reveal that the object's address controls the decision.", ha="center", fontsize=12.4, fontweight="bold")
export_presentation_figure(fig, "05_why_heatmaps_are_not_enough.png")

# J. Visual evidence ledger.
ledger_rows = [
    ("IID validation", "Does ordinary validation say it works?", "passes", "passes", "amber"),
    ("Crossed-position audit", "Does it work when position bias breaks?", "fails", "passes", "red"),
    ("Same-shape movement", "Does the same object keep its label?", "flips", "stable", "red"),
    ("Position-response map", "Does score follow canvas location?", "territory map", "shape-stable", "red"),
    ("Morph probe", "Does score follow shape or address?", "position-contaminated", "tracks shape", "amber"),
    ("Saliency", "Can heatmaps certify the reason?", "insufficient alone", "insufficient alone", "amber"),
    ("Intervention re-test", "Did changed incentives reduce risk?", "baseline risk", "improves", "green"),
]
traffic = {"red": "#C44E52", "green": "#2E8B57", "amber": "#D89C21"}
fig, axis = plt.subplots(figsize=(14.8, 7.4), constrained_layout=True)
axis.axis("off")
fig.suptitle("Shortcut evidence ledger", fontsize=19, fontweight="bold")
headers = ["Probe", "What it asks", "MLP evidence", "CNN evidence", "Verdict"]
col_x = [0.02, 0.20, 0.49, 0.68, 0.85]
col_w = [0.16, 0.27, 0.17, 0.15, 0.12]
for x0, width, header in zip(col_x, col_w, headers, strict=True):
    axis.add_patch(plt.Rectangle((x0, 0.90), width, 0.07, facecolor="#222222", transform=axis.transAxes))
    axis.text(x0 + 0.01, 0.922, header, color="white", fontsize=11.5, fontweight="bold", transform=axis.transAxes)
for row_index, (probe, question, mlp_evidence, cnn_evidence, verdict) in enumerate(ledger_rows):
    y0 = 0.82 - row_index * 0.105
    bg = "#FAFAFA" if row_index % 2 == 0 else "#F0F0F0"
    for x0, width in zip(col_x, col_w, strict=True):
        axis.add_patch(plt.Rectangle((x0, y0), width, 0.09, facecolor=bg, edgecolor="white", linewidth=1.0, transform=axis.transAxes))
    axis.text(col_x[0] + 0.01, y0 + 0.03, probe, fontsize=10.7, fontweight="bold", transform=axis.transAxes)
    axis.text(col_x[1] + 0.01, y0 + 0.03, question, fontsize=10.0, transform=axis.transAxes, wrap=True)
    axis.text(col_x[2] + 0.01, y0 + 0.03, mlp_evidence, fontsize=10.3, color=MODEL_COLOURS["MLP"], fontweight="bold", transform=axis.transAxes)
    axis.text(col_x[3] + 0.01, y0 + 0.03, cnn_evidence, fontsize=10.3, color=MODEL_COLOURS["CNN"], fontweight="bold", transform=axis.transAxes)
    axis.add_patch(plt.Circle((col_x[4] + 0.04, y0 + 0.045), 0.018, color=traffic[verdict], transform=axis.transAxes))
    axis.text(col_x[4] + 0.07, y0 + 0.031, verdict, fontsize=10.0, fontweight="bold", transform=axis.transAxes)
fig.text(0.5, -0.02, "No single probe proves the shortcut. The case is strong because behaviour, surfaces, morphs, saliency caveats, and re-test agree.", ha="center", fontsize=12.0, fontweight="bold")
export_presentation_figure(fig, "06_shortcut_evidence_ledger.png")


### Animation wall: the behavioural evidence

The cells above generate these assets under `outputs/notebook_figures/00_moons_stars_clever_hans/`. The notebook keeps code outputs clear for version control, so this markdown wall points directly at the generated files.

<div style="display:grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap:18px; align-items:start">
  <figure>
    <video src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_moon_moves_confidence.mp4" controls autoplay muted loop style="width:100%; border:1px solid #ddd"></video>
    <figcaption><strong>Moon movement:</strong> the same moon moves and the MLP answer changes.</figcaption>
  </figure>
  <figure>
    <video src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_star_moves_confidence.mp4" controls autoplay muted loop style="width:100%; border:1px solid #ddd"></video>
    <figcaption><strong>Star movement:</strong> the same star moves and the MLP answer changes.</figcaption>
  </figure>
  <figure>
    <img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_response_map_path_mlp.gif" style="width:100%; border:1px solid #ddd">
    <figcaption><strong>MLP response map:</strong> the same moon crosses into star territory.</figcaption>
  </figure>
  <figure>
    <img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_response_map_path_cnn.gif" style="width:100%; border:1px solid #ddd">
    <figcaption><strong>CNN response map:</strong> the same movement is much more shape-stable.</figcaption>
  </figure>
  <figure>
    <video src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_moon_moves_heatmaps.mp4" controls autoplay muted loop style="width:100%; border:1px solid #ddd"></video>
    <figcaption><strong>Moon movement heatmaps:</strong> saliency overlays update as supporting evidence.</figcaption>
  </figure>
  <figure>
    <video src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_morph_upper_right_heatmaps.mp4" controls autoplay muted loop style="width:100%; border:1px solid #ddd"></video>
    <figcaption><strong>Upper-right morph heatmaps:</strong> shape changes at the star-favoured address.</figcaption>
  </figure>
</div>


## 7. Act II: The CNN learns a different shortcut

After Act I it is tempting to conclude that CNNs solve the problem. That is the wrong conclusion. We have only removed one cheap rule. Now we give the CNN another cheap rule — one that humans barely see.

This is not adversarial magic. The cue is real but irrelevant. It is weak per pixel but strong when aggregated across the whole background.


In [ ]:
# M1. Act II constants and deterministic renderer.
BASE_GREY = 0.50
ACT2_NOISE_STD = 0.006
ACT2_CUE_CHANNEL = 0
ACT2_COLOUR_SCALE = 1500.0
ACT2_TINT_DELTA_CANDIDATES = [value / 255.0 for value in [1, 2, 3, 4, 5, 6, 8, 10]]
ACT2_SEED = SEED + 220_000
ACT2_COUNTERFACTUAL_SEED = ACT2_SEED + 92_000
ACT2_MARGIN_TARGET = 6.0
ACT2_SUCCESS_MESSAGE = "Act II failed: invisible background cue did not produce decisive confidence."


@dataclass(frozen=True)
class Act2Sample:
    image: Image.Image
    label: int
    background_style: str
    seed: int
    spec: ShapeSpec


def make_base_noise(seed: int, h: int = IMAGE_SIZE, w: int = IMAGE_SIZE) -> np.ndarray:
    local_rng = np.random.default_rng(seed)
    return local_rng.normal(0.0, ACT2_NOISE_STD, size=(h, w, 3)).astype(np.float32)


def render_subtle_background(style: str, seed: int, tint_delta: float, shape: tuple[int, int] = (IMAGE_SIZE, IMAGE_SIZE)) -> np.ndarray:
    background = BASE_GREY + make_base_noise(seed, shape[0], shape[1])
    if style == "star":
        background[..., ACT2_CUE_CHANNEL] += tint_delta
    return np.clip(background, 0.0, 1.0)


def interpolate_backgrounds(bg_a: np.ndarray, bg_b: np.ndarray, alpha: float) -> np.ndarray:
    return np.clip((1.0 - alpha) * bg_a + alpha * bg_b, 0.0, 1.0)


def make_act2_spec(label: int, seed: int, centre: tuple[int, int] | None = None) -> ShapeSpec:
    local_rng = rng_for(seed)
    chosen_centre = centre if centre is not None else choose_centre(FREE_REGION, local_rng)
    return ShapeSpec(
        label=label,
        centre=chosen_centre,
        seed=seed,
        size=float(local_rng.uniform(18.0, 22.0)),
        rotation=float(local_rng.uniform(-16.0, 16.0)),
        brightness=int(local_rng.integers(225, 256)),
        stroke=1,
        crescent=float(local_rng.uniform(0.62, 0.70)),
    )


def foreground_from_shape(spec: ShapeSpec) -> tuple[np.ndarray, np.ndarray]:
    grey = np.asarray(render_shape(spec), dtype=np.float32) / 255.0
    mask = grey > 0.34
    foreground = np.clip(0.18 + 0.82 * grey, 0.0, 1.0)
    return foreground, mask


def render_act2_image_from_spec(spec: ShapeSpec, background: np.ndarray) -> Image.Image:
    foreground, mask = foreground_from_shape(spec)
    rgb = background.copy()
    rgb[mask] = np.repeat(foreground[..., None], 3, axis=2)[mask]
    return Image.fromarray(np.uint8(np.clip(rgb, 0, 1) * 255), "RGB")


def render_same_shape_with_background_style(spec: ShapeSpec, background_style: str, seed: int, tint_delta: float) -> Image.Image:
    return render_act2_image_from_spec(spec, render_subtle_background(background_style, seed, tint_delta))


def render_neutral_act2_image(spec: ShapeSpec) -> Image.Image:
    neutral_background = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), 0.35, dtype=np.float32)
    return render_act2_image_from_spec(spec, neutral_background)


def make_invisible_background_dataset(
    count_per_class: int,
    *,
    tint_delta: float,
    cue_probability: float = 1.0,
    randomise_background: bool = False,
    neutralise_background: bool = False,
    seed: int = ACT2_SEED,
    split: str = "train",
) -> list[Act2Sample]:
    samples: list[Act2Sample] = []
    local_rng = rng_for(seed)
    for label in [0, 1]:
        for index in range(count_per_class):
            sample_seed = int(seed + label * 20_000 + index)
            spec = make_act2_spec(label, sample_seed)
            if neutralise_background:
                style = "neutralised"
                image = render_neutral_act2_image(spec)
            else:
                if randomise_background:
                    style = "star" if bool(local_rng.integers(0, 2)) else "moon"
                elif local_rng.random() < cue_probability:
                    style = CLASS_NAMES[label]
                else:
                    style = CLASS_NAMES[1 - label]
                image = render_same_shape_with_background_style(spec, style, sample_seed + 991, tint_delta)
            samples.append(Act2Sample(image=image, label=label, background_style=style, seed=sample_seed, spec=spec))
    local_rng.shuffle(samples)
    return samples


ACT2_TENSOR_CACHE: dict[tuple[int, ...], tuple[torch.Tensor, torch.Tensor]] = {}


def act2_samples_to_tensors(samples: list[Act2Sample]) -> tuple[torch.Tensor, torch.Tensor]:
    cache_key = tuple(id(sample) for sample in samples)
    if cache_key in ACT2_TENSOR_CACHE:
        return ACT2_TENSOR_CACHE[cache_key]
    arrays = [np.asarray(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0 for sample in samples]
    x = torch.tensor(np.stack(arrays).transpose(0, 3, 1, 2), dtype=torch.float32)
    y = torch.tensor([sample.label for sample in samples], dtype=torch.long)
    ACT2_TENSOR_CACHE[cache_key] = (x, y)
    return x, y


def image_batch_to_tensor(image_or_batch: Image.Image | list[Image.Image]) -> tuple[torch.Tensor, bool]:
    images = image_or_batch if isinstance(image_or_batch, list) else [image_or_batch]
    arrays = [np.asarray(image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0 for image in images]
    x = torch.tensor(np.stack(arrays).transpose(0, 3, 1, 2), dtype=torch.float32)
    return x, isinstance(image_or_batch, list)

In [ ]:
# M2. Act II models, confidence metrics, and margin training.
class Act2CueCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.visual = torch.nn.Sequential(
            torch.nn.Conv2d(3, 16, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(16, 32, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 48, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d(1),
        )
        self.colour_head = torch.nn.Sequential(
            torch.nn.Linear(3, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 16),
            torch.nn.ReLU(),
        )
        self.head = torch.nn.Linear(48 + 16, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        visual_features = self.visual(x).flatten(1)
        # Deliberately give the network access to global colour statistics.
        # This mirrors real acquisition colour, lighting, camera balance, batch effects, or background statistics.
        colour_stats = (x.mean(dim=(2, 3)) - 0.5) * ACT2_COLOUR_SCALE
        colour_features = self.colour_head(colour_stats)
        return self.head(torch.cat([visual_features, colour_features], dim=1))


class BackgroundMeanOnlyClassifier(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = torch.nn.Linear(3, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        colour_stats = (x.mean(dim=(2, 3)) - 0.5) * ACT2_COLOUR_SCALE
        return self.linear(colour_stats)


class Act2GrayGapCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.model = GapCNN()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        grey = x.mean(dim=1, keepdim=True)
        return self.model(grey)


class LogitScaledModel(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module, scale: float) -> None:
        super().__init__()
        self.base_model = base_model
        self.scale = float(scale)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base_model(x) * self.scale


def correct_class_probability(p_star: float, label: int) -> float:
    return float(p_star if label == 1 else 1.0 - p_star)


def margin_loss(logits: torch.Tensor, y: torch.Tensor, target_margin: float = ACT2_MARGIN_TARGET) -> torch.Tensor:
    correct = logits.gather(1, y[:, None]).squeeze(1)
    wrong = logits.gather(1, (1 - y)[:, None]).squeeze(1)
    return torch.relu(target_margin - (correct - wrong)).mean()


def predict_p_star(model: torch.nn.Module, image_or_batch: Image.Image | list[Image.Image]) -> float | np.ndarray:
    x, is_batch = image_batch_to_tensor(image_or_batch)
    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(x), dim=1)[:, 1].cpu().numpy()
    return probabilities if is_batch else float(probabilities[0])


def prediction_text(p_star: float) -> str:
    return f"star ({p_star:.2f})" if p_star >= 0.5 else f"moon ({1.0 - p_star:.2f})"


def make_confidence_bar(draw: ImageDraw.ImageDraw, xy: tuple[int, int], width: int, p_star: float, label: str, colour: tuple[int, int, int]) -> None:
    x, y = xy
    draw.rounded_rectangle((x, y, x + width, y + 28), radius=8, fill=(238, 238, 238), outline=(185, 185, 185))
    draw.rounded_rectangle((x, y, x + int(width * p_star), y + 28), radius=8, fill=colour)
    draw.text((x, y - 21), f"{label} P(star) {p_star:.2f} -> {prediction_text(p_star)}", fill=(30, 30, 30))


def evaluate_act2_model(model: torch.nn.Module, samples: list[Act2Sample]) -> dict[str, Any]:
    x, y = act2_samples_to_tensors(samples)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(logits, y).item()
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)
        star_scores = probabilities[:, 1].cpu().numpy()
    labels = y.cpu().numpy()
    correct_probs = np.array([correct_class_probability(float(score), int(label)) for score, label in zip(star_scores, labels, strict=True)])
    return {
        "loss": loss,
        "accuracy": float((predictions == y).float().mean().item()),
        "mean_correct_prob": float(correct_probs.mean()),
        "min_correct_prob": float(correct_probs.min()),
        "star_score": star_scores,
        "predictions": predictions.cpu().numpy(),
        "labels": labels,
    }


def train_act2_model(
    model: torch.nn.Module,
    train_samples: list[Act2Sample],
    validation_samples: list[Act2Sample],
    *,
    epochs: int = 200,
    learning_rate: float = 2e-3,
    require_confidence: bool = False,
) -> list[dict[str, float]]:
    x_train, y_train = act2_samples_to_tensors(train_samples)
    optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    history: list[dict[str, float]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimiser.zero_grad(set_to_none=True)
        logits = model(x_train)
        loss = torch.nn.functional.cross_entropy(logits, y_train) + 0.10 * margin_loss(logits, y_train)
        loss.backward()
        optimiser.step()
        train_eval = evaluate_act2_model(model, train_samples)
        validation_eval = evaluate_act2_model(model, validation_samples)
        history.append(
            {
                "epoch": float(epoch),
                "train_accuracy": train_eval["accuracy"],
                "validation_accuracy": validation_eval["accuracy"],
                "validation_mean_correct_prob": validation_eval["mean_correct_prob"],
                "validation_min_correct_prob": validation_eval["min_correct_prob"],
                "train_loss": train_eval["loss"],
                "validation_loss": validation_eval["loss"],
            }
        )
        if require_confidence:
            if (
                epoch >= 12
                and validation_eval["accuracy"] >= 0.995
                and validation_eval["mean_correct_prob"] >= 0.98
                and validation_eval["min_correct_prob"] >= 0.90
            ):
                break
        elif epoch >= 12 and validation_eval["accuracy"] >= 0.995:
            break
    return history


def calibrate_logit_scale(
    model: torch.nn.Module,
    validation_samples: list[Act2Sample],
    accept: Callable[[torch.nn.Module], bool],
    *,
    scales: tuple[float, ...] = (1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 16.0, 24.0, 32.0, 48.0, 64.0),
) -> tuple[torch.nn.Module, float]:
    for scale in scales:
        candidate = LogitScaledModel(model, scale)
        validation_eval = evaluate_act2_model(candidate, validation_samples)
        if validation_eval["accuracy"] >= 0.995 and accept(candidate):
            return candidate, scale
    return model, 1.0

In [ ]:
# M3. Calibrate the smallest invisible tint that gives decisive Act II confidence.
def make_act2_swap_cases(tint_delta: float) -> tuple[list[tuple[str, ShapeSpec, str, int, str]], dict[str, Image.Image]]:
    moon_spec = make_act2_spec(0, ACT2_SEED + 91_000, centre=(44, 72))
    star_spec = make_act2_spec(1, ACT2_SEED + 91_500, centre=(84, 54))
    cases = [
        ("same moon\nmoon background", moon_spec, "moon", 0, "moon_on_moon_bg"),
        ("same moon\nstar background", moon_spec, "star", 0, "moon_on_star_bg"),
        ("same star\nstar background", star_spec, "star", 1, "star_on_star_bg"),
        ("same star\nmoon background", star_spec, "moon", 1, "star_on_moon_bg"),
    ]
    seed_by_key = {
        "moon_on_moon_bg": ACT2_COUNTERFACTUAL_SEED,
        "moon_on_star_bg": ACT2_COUNTERFACTUAL_SEED,
        "star_on_star_bg": ACT2_COUNTERFACTUAL_SEED + 1,
        "star_on_moon_bg": ACT2_COUNTERFACTUAL_SEED + 1,
    }
    images = {
        key: render_same_shape_with_background_style(spec, style, seed_by_key[key], tint_delta)
        for _title, spec, style, _label, key in cases
    }
    return cases, images


def act2_swap_score_dict(model: torch.nn.Module, images: dict[str, Image.Image]) -> dict[str, float]:
    return {key: float(predict_p_star(model, image)) for key, image in images.items()}


def biased_swap_thresholds_pass(scores: dict[str, float]) -> bool:
    return (
        scores["moon_on_moon_bg"] <= 0.05
        and scores["moon_on_star_bg"] >= 0.95
        and scores["star_on_star_bg"] >= 0.95
        and scores["star_on_moon_bg"] <= 0.05
    )


def mitigated_swap_thresholds_pass(scores: dict[str, float]) -> bool:
    return (
        scores["moon_on_moon_bg"] <= 0.10
        and scores["moon_on_star_bg"] <= 0.10
        and scores["star_on_star_bg"] >= 0.90
        and scores["star_on_moon_bg"] >= 0.90
    )


def act2_ready(model: torch.nn.Module, validation_samples: list[Act2Sample], swap_images: dict[str, Image.Image]) -> bool:
    validation_eval = evaluate_act2_model(model, validation_samples)
    return (
        validation_eval["accuracy"] >= 0.995
        and validation_eval["mean_correct_prob"] >= 0.98
        and validation_eval["min_correct_prob"] >= 0.90
        and biased_swap_thresholds_pass(act2_swap_score_dict(model, swap_images))
    )


def train_background_only_for_delta(tint_delta: float) -> tuple[torch.nn.Module, list[Act2Sample], list[Act2Sample], list[dict[str, float]], dict[str, Any]]:
    train_samples = make_invisible_background_dataset(120, tint_delta=tint_delta, seed=ACT2_SEED, split="background-only train")
    validation_samples = make_invisible_background_dataset(50, tint_delta=tint_delta, seed=ACT2_SEED + 10_000, split="background-only validation")
    torch.manual_seed(ACT2_SEED + int(round(tint_delta * 255)) + 11)
    model = BackgroundMeanOnlyClassifier()
    history = train_act2_model(model, train_samples, validation_samples, epochs=160, learning_rate=2e-3, require_confidence=True)
    cases, swap_images = make_act2_swap_cases(tint_delta)
    def accept(candidate: torch.nn.Module) -> bool:
        return biased_swap_thresholds_pass(act2_swap_score_dict(candidate, swap_images))
    model, scale = calibrate_logit_scale(model, validation_samples, accept)
    evaluation = evaluate_act2_model(model, validation_samples)
    evaluation["scale"] = scale
    evaluation["swap_scores"] = act2_swap_score_dict(model, swap_images)
    return model, train_samples, validation_samples, history, evaluation


def train_cue_cnn_for_delta(tint_delta: float) -> dict[str, Any]:
    train_samples = make_invisible_background_dataset(160, tint_delta=tint_delta, seed=ACT2_SEED, split="train")
    validation_samples = make_invisible_background_dataset(60, tint_delta=tint_delta, seed=ACT2_SEED + 10_000, split="validation")
    cases, swap_images = make_act2_swap_cases(tint_delta)
    torch.manual_seed(ACT2_SEED + int(round(tint_delta * 255)) + 21)
    model = Act2CueCNN()
    history = train_act2_model(model, train_samples, validation_samples, epochs=200, learning_rate=2e-3, require_confidence=True)
    def accept(candidate: torch.nn.Module) -> bool:
        return act2_ready(candidate, validation_samples, swap_images)
    model, scale = calibrate_logit_scale(model, validation_samples, accept)
    validation_eval = evaluate_act2_model(model, validation_samples)
    swap_scores = act2_swap_score_dict(model, swap_images)
    return {
        "model": model,
        "scale": scale,
        "train_samples": train_samples,
        "validation_samples": validation_samples,
        "history": history,
        "validation_eval": validation_eval,
        "swap_cases": cases,
        "swap_images": swap_images,
        "swap_scores": swap_scores,
        "ready": act2_ready(model, validation_samples, swap_images),
    }


act2_calibration_trials: list[dict[str, Any]] = []
act2_background_only_trials: list[dict[str, Any]] = []
act2_selected_trial: dict[str, Any] | None = None
act2_background_only_model: torch.nn.Module | None = None
act2_background_only_eval: dict[str, Any] | None = None

for candidate_delta in ACT2_TINT_DELTA_CANDIDATES:
    bg_model, bg_train, bg_validation, bg_history, bg_eval = train_background_only_for_delta(candidate_delta)
    act2_background_only_trials.append(
        {
            "tint_delta": candidate_delta,
            "model": bg_model,
            "train_samples": bg_train,
            "validation_samples": bg_validation,
            "history": bg_history,
            "evaluation": bg_eval,
        }
    )
    trial = train_cue_cnn_for_delta(candidate_delta)
    trial["tint_delta"] = candidate_delta
    act2_calibration_trials.append(trial)
    if trial["ready"]:
        act2_selected_trial = trial
        act2_background_only_model = bg_model
        act2_background_only_eval = bg_eval
        break

if act2_selected_trial is None or act2_background_only_model is None or act2_background_only_eval is None:
    raise RuntimeError(ACT2_SUCCESS_MESSAGE)

ACT2_TINT_DELTA = float(act2_selected_trial["tint_delta"])
act2_biased_cnn = act2_selected_trial["model"]
act2_train_samples = act2_selected_trial["train_samples"]
act2_validation_samples = act2_selected_trial["validation_samples"]
act2_history = act2_selected_trial["history"]
act2_swap_cases = act2_selected_trial["swap_cases"]
act2_swap_images = act2_selected_trial["swap_images"]
act2_swap_scores = act2_selected_trial["swap_scores"]
act2_train_accuracy = evaluate_act2_model(act2_biased_cnn, act2_train_samples)["accuracy"]
act2_validation_eval = act2_selected_trial["validation_eval"]
act2_validation_accuracy = act2_validation_eval["accuracy"]
act2_validation_mean_correct_prob = act2_validation_eval["mean_correct_prob"]
act2_validation_min_correct_prob = act2_validation_eval["min_correct_prob"]
act2_background_only_accuracy = float(act2_background_only_eval["accuracy"])
act2_background_only_mean_correct_prob = float(act2_background_only_eval["mean_correct_prob"])
act2_background_only_min_correct_prob = float(act2_background_only_eval["min_correct_prob"])
act2_background_only_swap_scores = act2_background_only_eval["swap_scores"]
act2_tint_delta_label = f"{ACT2_TINT_DELTA * 255:.0f}/255"

print(f"Selected Act II tint delta: {act2_tint_delta_label}")
print(f"Background-only accuracy: {act2_background_only_accuracy:.3f}")
print(f"CNN IID accuracy: {act2_validation_accuracy:.3f}")
print(f"CNN mean correct-class probability: {act2_validation_mean_correct_prob:.3f}")
print(f"CNN minimum correct-class probability: {act2_validation_min_correct_prob:.3f}")
print(f"Swap scores: {act2_swap_scores}")

if not act2_ready(act2_biased_cnn, act2_validation_samples, act2_swap_images):
    raise RuntimeError(ACT2_SUCCESS_MESSAGE)

In [ ]:
# M4. Act II sanity check and apparent examples.
fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.8), constrained_layout=True)
fig.suptitle("The hidden cue alone can solve the biased exam", fontsize=18, fontweight="bold")
card_data = [
    ("Selected tint", act2_tint_delta_label, "small per-pixel shift", "#D89C21"),
    ("Background-only accuracy", f"{act2_background_only_accuracy * 100:.1f}%", f"mean correct prob {act2_background_only_mean_correct_prob:.2f}", MODEL_COLOURS["CNN"]),
    ("Minimum correct prob", f"{act2_background_only_min_correct_prob:.2f}", "hidden cue is decisive", MODEL_COLOURS["MLP"]),
]
for axis, (title, value, note, colour) in zip(axes, card_data, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.06, 0.12), 0.88, 0.74, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.4, transform=axis.transAxes))
    axis.text(0.5, 0.70, title, ha="center", fontsize=15, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.5, 0.45, value, ha="center", fontsize=25, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.5, 0.24, note, ha="center", fontsize=11.5, transform=axis.transAxes)
fig.text(0.5, -0.02, "This is the exam leak: the background statistic is weak per pixel, but strong when aggregated.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig20a_background_only_sanity_check.png")

fig, axes = plt.subplots(2, 4, figsize=(11.6, 6.2), constrained_layout=True)
fig.suptitle("Act II: the task still looks like shape recognition", fontsize=18, fontweight="bold")
fig.text(0.5, 0.92, "The backgrounds look the same to us.", ha="center", fontsize=13, color="#555555")
for axis, sample in zip(axes.flat, act2_train_samples[:8], strict=True):
    axis.imshow(sample.image)
    axis.set_title(CLASS_NAMES[sample.label], color=CLASS_COLOURS[sample.label], fontweight="bold")
    axis.set_xticks([])
    axis.set_yticks([])
fig.text(0.5, -0.02, "The cue is real in the pixels, but ordinary examples do not make it visually obvious.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig20_act2_apparent_shape_task_invisible_background.png")

fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.8), constrained_layout=True)
fig.suptitle("The CNN appears to work again", fontsize=18, fontweight="bold")
for axis, title, value, note, colour in [
    (axes[0], "train accuracy", act2_train_accuracy, "ordinary train", MODEL_COLOURS["CNN"]),
    (axes[1], "IID validation", act2_validation_accuracy, "repeats hidden cue", MODEL_COLOURS["CNN"]),
    (axes[2], "minimum correct prob", act2_validation_min_correct_prob, "not just argmax", "#D89C21"),
]:
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.08, 0.12), 0.84, 0.74, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.4, transform=axis.transAxes))
    axis.text(0.50, 0.65, title, ha="center", fontsize=15, fontweight="bold", transform=axis.transAxes)
    axis.text(0.50, 0.39, f"{value * 100:.1f}%" if "accuracy" in title else f"{value:.2f}", ha="center", fontsize=25, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.50, 0.22, note, ha="center", fontsize=11, transform=axis.transAxes)
fig.text(0.5, -0.02, "Everything still looks solved because validation repeats the hidden cue.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig21_act2_cnn_appears_to_work.png")

In [ ]:
# M5. Background-swap counterfactual, confidence sweep, and animations.
def decisive_badge_colour(score: float, label: int) -> str:
    prediction = 1 if score >= 0.5 else 0
    decisive = score <= 0.10 or score >= 0.90
    if prediction == label and decisive:
        return MODEL_COLOURS["CNN"]
    return "#B23A48"


def draw_act2_swap_card(
    canvas: Image.Image,
    draw: ImageDraw.ImageDraw,
    box: tuple[int, int, int, int],
    title: str,
    image: Image.Image,
    label: int,
    score: float,
) -> None:
    x0, y0, x1, y1 = box
    pred = int(score >= 0.5)
    ok = pred == label
    colour_hex = decisive_badge_colour(score, label)
    colour = tuple(int(colour_hex.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4))
    draw.rounded_rectangle((x0, y0, x1, y1), radius=22, fill=(250, 250, 250), outline=(220, 220, 220), width=2)
    draw.text((x0 + 24, y0 + 18), title.replace("\n", " / "), fill=(20, 20, 20), font=font_subtitle)
    draw.text((x0 + 24, y0 + 58), f"TRUE: {CLASS_NAMES[label]}", fill=(70, 70, 70), font=font_body)
    canvas.paste(image.resize((330, 330), Image.Resampling.LANCZOS).convert("RGB"), (x0 + 32, y0 + 96))
    draw.text((x0 + 405, y0 + 170), f"CNN: {CLASS_NAMES[pred]} | P(star)={score:.2f}", fill=colour, font=font_body_bold)
    draw_pil_confidence_bar(draw, (x0 + 405, y0 + 220), 420, float(score), colour, "intended evidence" if ok else "shortcut exposed", font=font_small_bold)

canvas = Image.new("RGB", (1900, 1420), (255, 255, 255))
draw = ImageDraw.Draw(canvas)
draw.text((130, 42), "Same shape. Almost invisible background shift. Different belief.", fill=(10, 10, 10), font=font_title)
act2_card_boxes = [(70, 130, 920, 625), (980, 130, 1830, 625), (70, 690, 920, 1185), (980, 690, 1830, 1185)]
for box, (title, _spec, _style, label, key) in zip(act2_card_boxes, act2_swap_cases, strict=True):
    draw_act2_swap_card(canvas, draw, box, title, act2_swap_images[key], label, act2_swap_scores[key])
if any(0.10 < score < 0.90 for score in act2_swap_scores.values()):
    draw.text((650, 1210), "DEMO FAILED: model not decisive", fill=(178, 58, 72), font=font_title)
draw.rounded_rectangle((105, 1235, 1795, 1302), radius=18, fill=(244, 244, 244), outline=(210, 210, 210), width=2)
draw.text((155, 1254), "Changed: calibrated 1/255 red-channel cue.  Unchanged: visible object, shape seed, intended label, and base noise field.", fill=(20, 20, 20), font=font_body_bold)
draw.text((330, 1342), "Only the tiny background tint changes within each pair; the base noise field is shared.", fill=(20, 20, 20), font=font_body_bold)
save_pil_presentation_image(canvas, "fig22_invisible_background_swap_counterfactual.png")

def render_act2_background_morph(spec: ShapeSpec, alpha: float, seed: int) -> Image.Image:
    moon_bg = render_subtle_background("moon", seed, ACT2_TINT_DELTA)
    star_bg = render_subtle_background("star", seed, ACT2_TINT_DELTA)
    return render_act2_image_from_spec(spec, interpolate_backgrounds(moon_bg, star_bg, alpha))


def act2_background_morph_frame(spec: ShapeSpec, alpha: float, title: str) -> Image.Image:
    image = render_act2_background_morph(spec, alpha, ACT2_SEED + 93_000)
    p_star = float(predict_p_star(act2_biased_cnn, image))
    frame = Image.new("RGB", (780, 430), (250, 250, 250))
    draw = ImageDraw.Draw(frame)
    draw.text((28, 24), title, fill=(25, 25, 25))
    draw.text((28, 54), f"background alpha: {alpha:.2f} | tint {act2_tint_delta_label}", fill=(80, 80, 80))
    frame.paste(image.resize((260, 260), Image.Resampling.LANCZOS), (42, 112))
    make_confidence_bar(draw, (360, 170), 340, p_star, "CNN", (46, 139, 87))
    draw.line((360, 270, 700, 270), fill=(35, 35, 35), width=3)
    draw.ellipse((360 + int(340 * alpha) - 7, 263, 360 + int(340 * alpha) + 7, 277), fill=(20, 20, 20))
    draw.text((360, 290), "moon-style background", fill=(75, 75, 75))
    draw.text((570, 290), "star-style background", fill=(75, 75, 75))
    if (spec.label == 0 and p_star >= 0.5) or (spec.label == 1 and p_star < 0.5):
        draw.rounded_rectangle((360, 330, 700, 372), radius=10, fill=(255, 232, 232), outline=(178, 58, 72), width=2)
        draw.text((378, 343), "prediction changed under background shift", fill=(140, 40, 45))
    return frame


act2_moon_spec = act2_swap_cases[0][1]
act2_star_spec = act2_swap_cases[2][1]
act2_bg_alphas = np.linspace(0, 1, 17)
act2_moon_bg_scores = np.array([float(predict_p_star(act2_biased_cnn, render_act2_background_morph(act2_moon_spec, float(alpha), ACT2_SEED + 93_000))) for alpha in act2_bg_alphas])
act2_star_bg_scores = np.array([float(predict_p_star(act2_biased_cnn, render_act2_background_morph(act2_star_spec, float(alpha), ACT2_SEED + 93_000))) for alpha in act2_bg_alphas])
act2_moon_frames = [act2_background_morph_frame(act2_moon_spec, float(alpha), "Fixed moon: background morphs moon-style -> star-style") for alpha in act2_bg_alphas]
act2_star_frames = [act2_background_morph_frame(act2_star_spec, float(alpha), "Fixed star: background morphs moon-style -> star-style") for alpha in act2_bg_alphas]
save_gif_or_warn(act2_moon_frames, "anim_invisible_background_morph_moon.gif", "anim_invisible_background_morph_moon.mp4", fps=8, duration_ms=120)
save_gif_or_warn(act2_star_frames, "anim_invisible_background_morph_star.gif", "anim_invisible_background_morph_star.mp4", fps=8, duration_ms=120)

fig, axis = plt.subplots(figsize=(9.6, 5.5), constrained_layout=True)
fig.suptitle("Confidence follows an almost invisible background cue", fontsize=18, fontweight="bold")
axis.plot(act2_bg_alphas, act2_moon_bg_scores, color=CLASS_COLOURS[0], linewidth=2.8, label="fixed moon")
axis.plot(act2_bg_alphas, act2_star_bg_scores, color=CLASS_COLOURS[1], linewidth=2.8, label="fixed star")
axis.axhline(0.5, color="black", linestyle="--", linewidth=1.0)
for scores, colour, label_text in [(act2_moon_bg_scores, CLASS_COLOURS[0], "moon flip"), (act2_star_bg_scores, CLASS_COLOURS[1], "star flip")]:
    crossing = first_crossing(act2_bg_alphas, scores, starts_above=bool(scores[0] >= 0.5))
    if crossing is not None:
        axis.axvline(crossing, color=colour, linestyle=":", linewidth=2.0)
        axis.text(crossing + 0.02, 0.08 if label_text.startswith("moon") else 0.82, f"{label_text} {crossing:.2f}", color=colour, fontweight="bold")
axis.set_xlabel("background interpolation: moon-style -> star-style")
axis.set_ylabel("CNN P(star)")
axis.set_ylim(-0.03, 1.03)
axis.grid(alpha=0.18)
axis.legend(frameon=False, loc="best")
export_presentation_figure(fig, "fig23_invisible_background_confidence_sweep.png")


In [ ]:
# M6. Amplified cue reveal and shape-vs-background diagnostics.
def make_difference_amplified_panel(bg_a: np.ndarray, bg_b: np.ndarray, amplification: float = 100.0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    diff = np.abs(bg_b - bg_a)
    amplified = np.clip(diff * amplification, 0, 1)
    return bg_a, bg_b, amplified


moon_bg_example = render_subtle_background("moon", ACT2_SEED + 94_000, ACT2_TINT_DELTA)
star_bg_example = render_subtle_background("star", ACT2_SEED + 94_000, ACT2_TINT_DELTA)
moon_bg, star_bg, amplified_diff = make_difference_amplified_panel(moon_bg_example, star_bg_example)
fig = plt.figure(figsize=(12.4, 6.2), constrained_layout=True)
grid = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.25])
fig.suptitle("The hidden cue was real — just not obvious to us", fontsize=18, fontweight="bold")
for axis, title, image in [
    (fig.add_subplot(grid[0, 0]), "moon-style background", moon_bg[:80, :80]),
    (fig.add_subplot(grid[0, 1]), "star-style background", star_bg[:80, :80]),
    (fig.add_subplot(grid[0, 2]), "absolute difference ×100", amplified_diff[:80, :80]),
]:
    axis.imshow(image)
    axis.set_title(title)
    axis.set_xticks([])
    axis.set_yticks([])
hist_axis = fig.add_subplot(grid[0, 3])
hist_axis.hist(moon_bg[..., ACT2_CUE_CHANNEL].ravel(), bins=32, alpha=0.62, color=CLASS_COLOURS[0], label="moon bg")
hist_axis.hist(star_bg[..., ACT2_CUE_CHANNEL].ravel(), bins=32, alpha=0.62, color=CLASS_COLOURS[1], label="star bg")
hist_axis.set_title("red-channel background values")
hist_axis.set_xlabel("pixel value")
hist_axis.set_ylabel("count")
hist_axis.legend(frameon=False)
fig.text(0.5, -0.02, "The per-pixel shift is tiny. Across thousands of background pixels, it becomes an easy statistical shortcut.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig24_invisible_background_difference_amplified.png")


def render_act2_shape_background(shape_alpha: float, background_alpha: float, centre: tuple[int, int], seed: int) -> Image.Image:
    grey = np.asarray(render_morph_image(shape_alpha, centre, seed), dtype=np.float32) / 255.0
    mask = grey > 0.34
    foreground = np.clip(0.18 + 0.82 * grey, 0.0, 1.0)
    moon_bg = render_subtle_background("moon", seed + 101, ACT2_TINT_DELTA)
    star_bg = render_subtle_background("star", seed + 101, ACT2_TINT_DELTA)
    rgb = interpolate_backgrounds(moon_bg, star_bg, background_alpha)
    rgb[mask] = np.repeat(foreground[..., None], 3, axis=2)[mask]
    return Image.fromarray(np.uint8(np.clip(rgb, 0, 1) * 255), "RGB")


act2_shape_alphas = np.linspace(0, 1, 15)
act2_background_alphas = np.linspace(0, 1, 15)
act2_background_surface = np.zeros((len(act2_background_alphas), len(act2_shape_alphas)))
for row, bg_alpha in enumerate(act2_background_alphas):
    images = [render_act2_shape_background(float(shape_alpha), float(bg_alpha), (64, 64), ACT2_SEED + 95_000) for shape_alpha in act2_shape_alphas]
    act2_background_surface[row, :] = predict_p_star(act2_biased_cnn, images)
fig, axis = plt.subplots(figsize=(8.4, 6.4), constrained_layout=True)
fig.suptitle("Does the CNN follow shape or background?", fontsize=18, fontweight="bold")
im = axis.imshow(act2_background_surface, cmap="RdYlBu_r", vmin=0, vmax=1, origin="lower", extent=[0, 1, 0, 1], aspect="auto", interpolation="bicubic")
axis.contour(act2_shape_alphas, act2_background_alphas, act2_background_surface, levels=[0.5], colors="black", linewidths=1.8)
axis.set_xlabel("shape morph: moon -> star")
axis.set_ylabel("background tint: moon-style -> star-style")
fig.colorbar(im, ax=axis, label="CNN P(star)")
fig.text(0.5, -0.02, "If the surface changes mainly with background alpha, the hidden cue is controlling the score.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig25_background_tint_response_surface.png")

act2_bar_cases = [
    ("moon shape\nmoon bg", 0.0, 0.0),
    ("moon shape\nstar bg", 0.0, 1.0),
    ("half morph\nmoon bg", 0.5, 0.0),
    ("half morph\nstar bg", 0.5, 1.0),
    ("star shape\nmoon bg", 1.0, 0.0),
    ("star shape\nstar bg", 1.0, 1.0),
]
act2_bar_scores = [float(predict_p_star(act2_biased_cnn, render_act2_shape_background(shape_alpha, bg_alpha, (64, 64), ACT2_SEED + 96_000 + idx))) for idx, (_label, shape_alpha, bg_alpha) in enumerate(act2_bar_cases)]
fig, axis = plt.subplots(figsize=(10.6, 5.4), constrained_layout=True)
fig.suptitle("Shape changed less than background did", fontsize=18, fontweight="bold")
bars = axis.bar(np.arange(len(act2_bar_cases)), act2_bar_scores, color=[CLASS_COLOURS[0], CLASS_COLOURS[1], "#999999", CLASS_COLOURS[1], CLASS_COLOURS[0], CLASS_COLOURS[1]])
axis.axhline(0.5, color="black", linestyle="--", linewidth=1.0)
axis.set_ylim(0, 1.05)
axis.set_ylabel("CNN P(star)")
axis.set_xticks(np.arange(len(act2_bar_cases)))
axis.set_xticklabels([case[0] for case in act2_bar_cases])
for bar, score in zip(bars, act2_bar_scores, strict=True):
    axis.text(bar.get_x() + bar.get_width() / 2, score + 0.03, f"{score:.2f}", ha="center", fontweight="bold")
export_presentation_figure(fig, "fig26_background_vs_shape_bars.png")

In [ ]:
# M7. Compact saliency caution, mitigation, and re-test.
def act2_smoothgrad_saliency(model: torch.nn.Module, image: Image.Image, target_class: int = 1, samples: int = 2, noise: float = 0.02) -> np.ndarray:
    model.eval()
    array = np.asarray(image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
    base = torch.tensor(array.transpose(2, 0, 1), dtype=torch.float32).unsqueeze(0)
    saliency = torch.zeros_like(base)
    for _ in range(samples):
        noisy = (base + torch.randn_like(base) * noise).clamp(0, 1).requires_grad_(True)
        score = model(noisy)[0, target_class]
        model.zero_grad(set_to_none=True)
        score.backward()
        saliency += noisy.grad.detach().abs()
    saliency = saliency.mean(dim=1).squeeze().cpu().numpy() / samples
    return saliency / max(float(saliency.max()), 1e-8)


act2_caution_image = act2_swap_images["moon_on_star_bg"]
act2_caution_saliency = act2_smoothgrad_saliency(act2_biased_cnn, act2_caution_image)
fig, axes = plt.subplots(1, 4, figsize=(13.0, 4.3), constrained_layout=True)
fig.suptitle("Even plausible heatmaps are not enough", fontsize=18, fontweight="bold")
axes[0].imshow(act2_caution_image)
axes[0].set_title("same moon\nstar-style bg")
axes[1].imshow(act2_caution_image.resize((MODEL_SIZE, MODEL_SIZE)))
axes[1].imshow(act2_caution_saliency, cmap="magma", alpha=0.68)
axes[1].set_title("SmoothGrad")
axes[2].bar(["moon bg", "star bg"], [act2_swap_scores["moon_on_moon_bg"], act2_swap_scores["moon_on_star_bg"]], color=[CLASS_COLOURS[0], CLASS_COLOURS[1]])
axes[2].axhline(0.5, color="black", linestyle="--", linewidth=1.0)
axes[2].set_ylim(0, 1)
axes[2].set_title("background swap\nP(star)")
axes[3].imshow(amplified_diff[:80, :80])
axes[3].set_title("amplified\ndifference")
for axis in [axes[0], axes[1], axes[3]]:
    axis.set_xticks([])
    axis.set_yticks([])
fig.text(0.5, -0.02, "The decisive evidence is the counterfactual: change the irrelevant background cue and the model changes its mind.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig27_act2_heatmaps_are_not_enough.png")


def train_shape_mitigation_variant(name: str, *, randomise_background: bool, neutralise_background: bool, seed_offset: int) -> dict[str, Any]:
    train_samples = make_invisible_background_dataset(
        160,
        tint_delta=ACT2_TINT_DELTA,
        randomise_background=randomise_background,
        neutralise_background=neutralise_background,
        seed=ACT2_SEED + seed_offset,
        split=f"{name} train",
    )
    validation_samples = make_invisible_background_dataset(
        60,
        tint_delta=ACT2_TINT_DELTA,
        randomise_background=randomise_background,
        neutralise_background=neutralise_background,
        seed=ACT2_SEED + seed_offset + 10_000,
        split=f"{name} validation",
    )
    torch.manual_seed(ACT2_SEED + seed_offset + 33)
    model = Act2GrayGapCNN()
    history = train_act2_model(model, train_samples, validation_samples, epochs=200, learning_rate=2e-3, require_confidence=True)
    def score_candidate(candidate: torch.nn.Module) -> dict[str, float]:
        return {
            key: float(predict_p_star(candidate, image))
            for key, image in act2_swap_images.items()
        }
    def accept(candidate: torch.nn.Module) -> bool:
        return mitigated_swap_thresholds_pass(score_candidate(candidate))
    model, scale = calibrate_logit_scale(model, validation_samples, accept)
    validation_eval = evaluate_act2_model(model, validation_samples)
    swap_scores = score_candidate(model)
    return {
        "name": name,
        "model": model,
        "scale": scale,
        "train_samples": train_samples,
        "validation_samples": validation_samples,
        "history": history,
        "validation_eval": validation_eval,
        "swap_scores": swap_scores,
        "ready": validation_eval["accuracy"] >= 0.90 and mitigated_swap_thresholds_pass(swap_scores),
    }


act2_mitigation_trials = [
    train_shape_mitigation_variant("background randomisation", randomise_background=True, neutralise_background=False, seed_offset=20_000),
    train_shape_mitigation_variant("background standardisation", randomise_background=False, neutralise_background=True, seed_offset=40_000),
]
act2_ready_mitigation_trials = [trial for trial in act2_mitigation_trials if trial["ready"]]
if not act2_ready_mitigation_trials:
    raise RuntimeError("Act II failed: mitigation did not produce decisive shape-following swap scores.")
act2_mitigation_trial = max(
    act2_ready_mitigation_trials,
    key=lambda trial: (trial["validation_eval"]["accuracy"], trial["validation_eval"]["mean_correct_prob"]),
)
act2_mitigated_cnn = act2_mitigation_trial["model"]
act2_mitigated_validation_accuracy = act2_mitigation_trial["validation_eval"]["accuracy"]
act2_mitigated_validation_mean_correct_prob = act2_mitigation_trial["validation_eval"]["mean_correct_prob"]
act2_mitigated_validation_min_correct_prob = act2_mitigation_trial["validation_eval"]["min_correct_prob"]
act2_mitigated_swap_scores = act2_mitigation_trial["swap_scores"]
act2_mitigation_strategy = act2_mitigation_trial["name"]
act2_biased_swap_gap = float(abs(act2_swap_scores["moon_on_star_bg"] - act2_swap_scores["moon_on_moon_bg"]) + abs(act2_swap_scores["star_on_star_bg"] - act2_swap_scores["star_on_moon_bg"])) / 2.0
act2_mitigated_swap_gap = float(abs(act2_mitigated_swap_scores["moon_on_star_bg"] - act2_mitigated_swap_scores["moon_on_moon_bg"]) + abs(act2_mitigated_swap_scores["star_on_star_bg"] - act2_mitigated_swap_scores["star_on_moon_bg"])) / 2.0

fig, axes = plt.subplots(2, 4, figsize=(14.2, 6.4), constrained_layout=True)
fig.suptitle("The fix is process change plus re-test", fontsize=18, fontweight="bold")
for col, (_title, _spec, _style, label, key) in enumerate(act2_swap_cases):
    biased_score = act2_swap_scores[key]
    mitigated_score = act2_mitigated_swap_scores[key]
    for row, model_name, score, colour in [
        (0, "biased CNN", biased_score, "#B23A48"),
        (1, "mitigated CNN", mitigated_score, MODEL_COLOURS["CNN"]),
    ]:
        axis = axes[row, col]
        axis.axis("off")
        axis.add_patch(plt.Rectangle((0.06, 0.12), 0.88, 0.74, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.1, transform=axis.transAxes))
        axis.text(0.5, 0.72, model_name, ha="center", fontsize=12.5, fontweight="bold", color=colour, transform=axis.transAxes)
        axis.text(0.5, 0.47, key.replace("_", "\n"), ha="center", fontsize=9.5, transform=axis.transAxes)
        axis.text(0.5, 0.24, f"P(star)={score:.2f}", ha="center", fontsize=13, fontweight="bold", transform=axis.transAxes)
fig.text(0.5, -0.02, f"Mitigation selected: {act2_mitigation_strategy}. Do not trust the architecture; change the data/process, then re-test behaviour.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig28_act2_mitigation_retest.png")

In [ ]:
# M8. Two-act evidence board, final XAI loop, bridge, and presentation mode.
board_rows = [
    ("Act I IID validation", "both models appear strong", "ordinary validation", "test repeated position bias", "amber"),
    ("Act I position movement", "same object should keep label", "same-shape movement", "MLP learned address", "red"),
    ("Act I response map", "decision should follow shape", "scan object position", "MLP learned geography", "red"),
    ("Act I intervention", "CNN should reduce shortcut", "position augmentation re-test", "more shape-stable", "green"),
    ("Act II IID validation", "CNN appears strong", "ordinary validation", "test repeated background cue", "amber"),
    ("Act II background swap", "same shape should keep label", "swap invisible cue", "CNN used background", "red"),
    ("Act II confidence sweep", "score should follow shape", "morph background", "confidence follows cue", "red"),
    ("Act II mitigation", "fix should reduce cue reliance", "background intervention", "sensitivity reduced", "green"),
]
colour_map = {"green": "#2E8B57", "red": "#B23A48", "amber": "#D89C21"}
fig, axes = plt.subplots(len(board_rows), 4, figsize=(14.6, 9.2), constrained_layout=True)
fig.suptitle("XAI as behavioural observability: reveal, revise, re-test", fontsize=18, fontweight="bold")
headers = ["What appeared true", "What XAI tested", "What was actually learned", "What changed after intervention"]
for col, header in enumerate(headers):
    axes[0, col].text(0.5, 1.18, header, ha="center", va="bottom", fontsize=11.5, fontweight="bold", transform=axes[0, col].transAxes)
for row_idx, (row_title, appeared, tested, learned, status) in enumerate(board_rows):
    values = [f"{row_title}\n{appeared}", tested, learned, "re-test" if row_idx in {3, 7} else "risk remains"]
    for col_idx, value in enumerate(values):
        axis = axes[row_idx, col_idx]
        axis.axis("off")
        colour = colour_map[status] if col_idx == 2 else "#D0D0D0"
        axis.add_patch(plt.Rectangle((0.02, 0.12), 0.96, 0.76, facecolor="#FAFAFA", edgecolor=colour, linewidth=2.0, transform=axis.transAxes))
        axis.text(0.5, 0.50, value, ha="center", va="center", fontsize=9.2, transform=axis.transAxes, wrap=True)
fig.text(0.5, -0.01, "Accuracy is not behaviour. The shortcut changes. The XAI discipline does not.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig29_two_act_evidence_board.png")

# Final XAI loop.
loop_steps = [
    ("Observe", "ordinary validation\nand hard-case behaviour"),
    ("Hypothesise", "what cheap rule\ncould explain it?"),
    ("Intervene", "change data, process,\nor model incentive"),
    ("Re-test", "same counterfactuals\nafter the fix"),
    ("Monitor", "watch for drift and\nnew shortcuts"),
]
fig, axis = plt.subplots(figsize=(13.2, 5.8), constrained_layout=True)
axis.axis("off")
fig.suptitle("The XAI loop: make learned behaviour observable", fontsize=19, fontweight="bold")
xs = np.linspace(0.10, 0.90, len(loop_steps))
for idx, ((step, note), x) in enumerate(zip(loop_steps, xs, strict=True)):
    colour = ["#4C78A8", "#D89C21", "#B23A48", "#2E8B57", "#777777"][idx]
    axis.add_patch(plt.Circle((x, 0.58), 0.085, facecolor="white", edgecolor=colour, linewidth=3.0, transform=axis.transAxes))
    axis.text(x, 0.60, step, ha="center", va="center", fontsize=12.5, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(x, 0.33, note, ha="center", va="center", fontsize=10.8, transform=axis.transAxes)
    if idx < len(xs) - 1:
        axis.annotate("", xy=(xs[idx + 1] - 0.10, 0.58), xytext=(x + 0.10, 0.58), xycoords=axis.transAxes, arrowprops={"arrowstyle": "->", "lw": 2.2, "color": "#444444"})
for x0, width, label, colour, size in [
    (0.06, 0.24, "Act I: position shortcut", MODEL_COLOURS["MLP"], 9.6),
    (0.36, 0.28, "Act II: invisible acquisition\n/background cue", "#B23A48", 9.2),
    (0.69, 0.27, "Later demos:\nWaterbirds, industrial markers, PatchCore, drift", "#555555", 8.5),
]:
    axis.add_patch(plt.Rectangle((x0, 0.045), width, 0.095, facecolor="#FAFAFA", edgecolor=colour, linewidth=1.5, transform=axis.transAxes))
    axis.text(x0 + 0.012, 0.092, label, fontsize=size, fontweight="bold", color=colour, va="center", transform=axis.transAxes)
fig.text(0.5, -0.02, "The loop is the reusable method: observe, test a hypothesis, intervene, and re-test the same behaviour.", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig_final_xai_loop.png")

# Real-world bridge cards.
bridge_cards = [
    ("Position shortcut", "object placement\ncrop geometry\nscanner alignment", "Act I movement counterfactual"),
    ("Invisible tint", "lighting\ncamera balance\nbatch effects\nacquisition pipeline", "Act II background swap"),
    ("Marker / border shortcut", "fixture marks\ntimestamps\nlabels\nwatermarks", "industrial side-band demo"),
    ("Wrong normality", "PatchCore memory-bank\ncontamination", "nearest-normal provenance"),
]
fig, axes = plt.subplots(1, 4, figsize=(14.4, 5.2), constrained_layout=True)
fig.suptitle("How the toy shortcuts map to real XAI risks", fontsize=19, fontweight="bold")
for axis, (title, examples, diagnostic) in zip(axes, bridge_cards, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.05, 0.10), 0.90, 0.78, facecolor="#FAFAFA", edgecolor="#333333", linewidth=1.8, transform=axis.transAxes))
    axis.text(0.50, 0.76, title, ha="center", fontsize=14, fontweight="bold", transform=axis.transAxes)
    axis.text(0.50, 0.48, examples, ha="center", va="center", fontsize=12, transform=axis.transAxes)
    axis.text(0.50, 0.18, diagnostic, ha="center", fontsize=10.5, color="#555555", fontweight="bold", transform=axis.transAxes)
fig.text(0.5, -0.02, "The surface cue changes; the audit question remains: what evidence changes the decision?", ha="center", fontsize=12, fontweight="bold")
export_presentation_figure(fig, "fig_final_real_world_bridge.png")

# Existing broader bridge retained for the full notebook.
bridge_rows = [
    ("Toy position shortcut", "Acquisition geometry / crop / object placement bias", "A cheap spatial rule can masquerade as object recognition."),
    ("Invisible background tint", "Waterbirds habitat / camera colour balance / lighting / batch effects", "A weak irrelevant cue can dominate when validation repeats it."),
    ("Corner or border artefact", "Industrial fixture marks / scanner edges / labels / timestamps", "The model can learn the process, not the defect."),
    ("Wrong normality", "PatchCore memory-bank contamination", "Nearest-normal provenance matters because memory can learn the wrong normal."),
    ("Confidence sweep", "What changes the decision", "Behavioural probes make learned evidence observable."),
    ("Mitigation and re-test", "Assurance lifecycle", "Every fix needs the same hard-case re-test."),
]
fig, axes = plt.subplots(len(bridge_rows), 1, figsize=(13.8, 8.4), constrained_layout=True)
fig.suptitle("From toy laboratory to real XAI audits", fontsize=19, fontweight="bold")
for axis, (lab_signal, real_system, lesson) in zip(axes, bridge_rows, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.02, 0.14), 0.24, 0.70, facecolor="#EEF3FA", edgecolor=CLASS_COLOURS[0], linewidth=1.8, transform=axis.transAxes))
    axis.add_patch(plt.Rectangle((0.38, 0.14), 0.26, 0.70, facecolor="#FFF5E6", edgecolor=CLASS_COLOURS[1], linewidth=1.8, transform=axis.transAxes))
    axis.add_patch(plt.Rectangle((0.74, 0.14), 0.24, 0.70, facecolor="#F6F6F6", edgecolor="#777777", linewidth=1.5, transform=axis.transAxes))
    axis.annotate("", xy=(0.37, 0.49), xytext=(0.27, 0.49), xycoords=axis.transAxes, arrowprops={"arrowstyle": "->", "lw": 2.0, "color": "#444444"})
    axis.annotate("", xy=(0.73, 0.49), xytext=(0.65, 0.49), xycoords=axis.transAxes, arrowprops={"arrowstyle": "->", "lw": 2.0, "color": "#444444"})
    axis.text(0.04, 0.50, lab_signal, va="center", fontsize=12.2, fontweight="bold", color=CLASS_COLOURS[0], transform=axis.transAxes)
    axis.text(0.40, 0.50, real_system, va="center", fontsize=12.2, fontweight="bold", color="#9A5B00", transform=axis.transAxes)
    axis.text(0.76, 0.50, lesson, va="center", fontsize=10.6, transform=axis.transAxes, wrap=True)
fig.text(0.5, -0.015, "The controlled shortcut is small; the audit logic is the reusable part.", ha="center", fontsize=12.2, fontweight="bold")
export_presentation_figure(fig, "fig_final_bridge_to_real_xai.png")

suggested_slide_order = [
    "Title: What Did the Model Actually Learn?",
    "The apparent task: moon versus star",
    "Two models: PixelMLP and CNN",
    "Both models appear to work",
    "Same moon, different place, different answer",
    "Movement animation and confidence path",
    "Hidden position shortcut revealed",
    "Position response maps: the model learned geography",
    "Act I verdict",
    "Act II: the task still looks like shape recognition",
    "CNN again appears to work",
    "Same shape, invisible background shift, different belief",
    "Background morph animation",
    "Confidence follows the hidden cue",
    "Amplified difference: the hidden cue was real",
    "Shape-versus-background surface",
    "Mitigation and re-test",
    "Two-act evidence board",
    "XAI loop and bridge to real systems",
    "Closing: accuracy is not behaviour.",
]
slide_manifest = {
    "fig00_apparent_shape_task.png": "Use this as slide 2: innocent apparent task.",
    "fig00_model_cards.png": "Use this as slide 3: model incentives.",
    "fig01_both_models_appear_to_work.png": "Use this as slide 4: apparent success.",
    "01_same_shape_movement_counterfactual.png": "Use this as slide 5: Act I reveal.",
    "anim_moon_moves_confidence.mp4": "Use this as slide 6: animated moon movement reveal.",
    "02_movement_confidence_paths.png": "Use this as slide 6 backup: static confidence path.",
    "00_hidden_position_shortcut.png": "Use this as slide 7: hidden exam leak.",
    "03_position_response_maps_with_boundaries.png": "Use this as slide 8: model geography.",
    "06_shortcut_evidence_ledger.png": "Use this as slide 9: Act I verdict.",
    "fig20_act2_apparent_shape_task_invisible_background.png": "Use this as slide 10: Act II apparent task.",
    "fig21_act2_cnn_appears_to_work.png": "Use this as slide 11: CNN appears to work.",
    "fig22_invisible_background_swap_counterfactual.png": "Use this as slide 12: Act II reveal.",
    "anim_invisible_background_morph_moon.mp4": "Use this as slide 13: invisible background morph.",
    "fig23_invisible_background_confidence_sweep.png": "Use this as slide 14: confidence follows cue.",
    "fig24_invisible_background_difference_amplified.png": "Use this as slide 15: amplified cue reveal.",
    "fig25_background_tint_response_surface.png": "Use this as slide 16: shape versus background.",
    "fig28_act2_mitigation_retest.png": "Use this as slide 17: mitigation and re-test.",
    "fig29_two_act_evidence_board.png": "Use this as slide 18: two-act evidence board.",
    "fig_final_xai_loop.png": "Use this as slide 19: reusable XAI loop.",
    "fig_final_real_world_bridge.png": "Use this as slide 19 backup: real-world bridge cards.",
}
manifest_lines = ["### Presentation export manifest", ""]
for filename, usage in slide_manifest.items():
    rel_path = presentation_export_manifest.get(filename, str((FIGURE_DIR / filename).relative_to(PROJECT_ROOT)))
    manifest_lines.append(f"- `{rel_path}` — {usage}")
manifest_lines.extend(["", "### Suggested slide order", ""])
for index, item in enumerate(suggested_slide_order, start=1):
    manifest_lines.append(f"{index}. {item}")

def presentation_asset(filename: str, caption: str, *, media: str = "image") -> str:
    rel_path = presentation_export_manifest[filename]
    if media == "video":
        body = f"<video src='../../{rel_path}' controls autoplay muted loop style='width:100%; border:1px solid #ddd'></video>"
    else:
        body = f"<img src='../../{rel_path}' style='width:100%; border:1px solid #ddd'>"
    return f"<figure style='margin:0 0 22px 0'>{body}<figcaption style='font-size:14px; font-weight:650; margin-top:6px'>{caption}</figcaption></figure>"

presentation_mode_assets = [
    ("fig00_apparent_shape_task.png", "1. The apparent task", "image"),
    ("fig00_model_cards.png", "2. Two models", "image"),
    ("fig01_both_models_appear_to_work.png", "3. Both models appear to work", "image"),
    ("01_same_shape_movement_counterfactual.png", "4. Act I reveal: same object, different address", "image"),
    ("anim_moon_moves_confidence.mp4", "5. Act I animation: movement changes confidence", "video"),
    ("02_movement_confidence_paths.png", "6. How far before the MLP changes its mind?", "image"),
    ("00_hidden_position_shortcut.png", "7. Reveal the hidden position shortcut", "image"),
    ("03_position_response_maps_with_boundaries.png", "8. The model learned geography", "image"),
    ("06_shortcut_evidence_ledger.png", "9. Act I verdict", "image"),
    ("fig20_act2_apparent_shape_task_invisible_background.png", "10. Act II still looks innocent", "image"),
    ("fig21_act2_cnn_appears_to_work.png", "11. CNN appears to work again", "image"),
    ("fig22_invisible_background_swap_counterfactual.png", "12. Act II reveal: invisible cue changes belief", "image"),
    ("anim_invisible_background_morph_moon.mp4", "13. Act II animation: cue morph changes confidence", "video"),
    ("fig23_invisible_background_confidence_sweep.png", "14. Confidence follows the hidden cue", "image"),
    ("fig24_invisible_background_difference_amplified.png", "15. The cue was real", "image"),
    ("fig25_background_tint_response_surface.png", "16. Shape versus background", "image"),
    ("fig28_act2_mitigation_retest.png", "17. Mitigation and re-test", "image"),
    ("fig29_two_act_evidence_board.png", "18. Two-act evidence board", "image"),
    ("fig_final_xai_loop.png", "19. The reusable XAI loop", "image"),
    ("fig_final_real_world_bridge.png", "20. Bridge to real systems", "image"),
]
display(HTML(
    "<h2>Presentation mode: essential story only</h2>"
    "<p><strong>Accuracy is not behaviour.</strong> The sequence below is the slide-ready route through the notebook.</p>"
    + "".join(presentation_asset(filename, caption, media=media) for filename, caption, media in presentation_mode_assets)
))
display(Markdown("\n".join(manifest_lines)))


### 7.1 The task still looks like shape recognition

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig20_act2_apparent_shape_task_invisible_background.png" style="width:100%; border:1px solid #ddd">

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig21_act2_cnn_appears_to_work.png" style="width:100%; border:1px solid #ddd">


### 7.2 Same shape, almost invisible background shift

<div style="border-left:8px solid #C44E52; background:#FFF1F2; padding:18px 22px; margin:16px 0; font-size:22px; font-weight:800;">
Same shape. Almost invisible background shift. Different belief.
</div>

Changed: a `1/255` red-channel background cue selected by calibration. Unchanged: visible object, shape seed, intended label, and base noise field within each swap pair.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig22_invisible_background_swap_counterfactual.png" style="width:100%; border:1px solid #ddd">

<video src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_invisible_background_morph_moon.mp4" controls autoplay muted loop style="width:49%; border:1px solid #ddd"></video>
<video src="../../outputs/notebook_figures/00_moons_stars_clever_hans/anim_invisible_background_morph_star.mp4" controls autoplay muted loop style="width:49%; border:1px solid #ddd"></video>


### 7.3 Reveal, diagnose, mitigate, re-test

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig23_invisible_background_confidence_sweep.png" style="width:100%; border:1px solid #ddd">

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig24_invisible_background_difference_amplified.png" style="width:100%; border:1px solid #ddd">

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig25_background_tint_response_surface.png" style="width:49%; border:1px solid #ddd">
<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig28_act2_mitigation_retest.png" style="width:49%; border:1px solid #ddd">

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig29_two_act_evidence_board.png" style="width:100%; border:1px solid #ddd">


## 11. Final evidence board and bridge

This was a toy laboratory. The same pattern appears in real systems.

<img src="../../outputs/notebook_figures/00_moons_stars_clever_hans/fig_final_bridge_to_real_xai.png" style="width:100%; border:1px solid #ddd">


## 12. What we learned

- The visible task was moons versus stars.
- IID validation made both models look strong.
- Moving the same moon or star exposed that the MLP had learned a positional rule.
- The hidden training distribution made that rule cheap and predictive.
- Position-response maps made the learned geography visible.
- Shape-morph and movement-path probes showed what changed the model score.
- Saliency alone was not enough; it had to be paired with behavioural counterfactuals.
- Act II showed that the CNN can learn a different cheap cue when the data makes it predictive.
- The translation-aware CNN with position augmentation learned a more shape-stable rule in this controlled setting.


## Residual risks and next questions

- This is generated controlled data, not real-world proof.
- The CNN is more translation-robust here, not perfectly translation invariant.
- Saliency, relevance density, representation neighbours, and probes are diagnostic evidence, not causal proof.
- Real datasets can contain several entangled shortcuts at once.
- Later notebooks repeat the pattern on industrial images, Waterbirds, PatchCore anomaly provenance, and explanation drift.


## Final audit verdict

### What this demo shows

✅ A model can look accurate while learning a position shortcut.  
✅ IID validation can hide the failure.  
✅ Same-shape movement exposes whether the model learned where or what.  
✅ Position-response maps reveal a location-dependent decision rule.  
✅ Shape-morph strips show whether the score follows shape or position.  
✅ Movement-path probes show how far an object must move before the model changes its mind.  
✅ Saliency alone is not enough; it must be paired with behavioural counterfactuals.  
✅ A translation-aware CNN with position augmentation reduces the position shortcut in Act I.  
✅ Act II shows the CNN can still learn an almost invisible background cue.  
✅ Mitigation means changing the data/process and re-testing the learned behaviour.

### What this demo does not prove

⚠️ This is a controlled generated demo, not real-world proof.  
⚠️ CNNs are not automatically perfectly translation invariant.  
⚠️ Occlusion, saliency, relevance density, and representation probes are diagnostic evidence, not causal proof.  
⚠️ Real datasets can contain several entangled shortcuts at once.  
⚠️ Later notebooks repeat the same audit pattern on industrial images, Waterbirds, PatchCore, and explanation drift.

<div style="border-left:8px solid #222; background:#F4F4F4; padding:18px 22px; margin:16px 0; font-size:22px; font-weight:800;">
The shortcut changes. The XAI discipline does not.
</div>

XAI is not a heatmap. Accuracy tells us whether the model was right on the exam. XAI asks whether it learned the rule we meant to teach.


In [ ]:
assert DATA_MODE == "generated_controlled_demo"
assert EXTERNAL_DATA_REQUIRED is False
assert IMAGE_SIZE == 128
assert len(COMMON_GROUPS) == 2
assert len(CROSSED_GROUPS) == 2

assert mlp_train_accuracy >= 0.995
assert mlp_iid_validation_accuracy >= 0.995
assert mlp_iid_validation_mean_correct_prob >= 0.98
assert mlp_iid_validation_min_correct_prob >= 0.90
assert mlp_ood_accuracy < 0.70
assert mlp_worst_group_accuracy < 0.50
assert mlp_crossed_accuracy <= 0.35
assert mlp_counterfactual_flip_rate > 0.50

assert cnn_iid_validation_accuracy >= 0.995
assert cnn_iid_validation_mean_correct_prob >= 0.98
assert cnn_iid_validation_min_correct_prob >= 0.90
assert cnn_ood_accuracy > 0.85
assert cnn_worst_group_accuracy > mlp_worst_group_accuracy
assert cnn_crossed_accuracy >= 0.90
assert cnn_counterfactual_flip_rate < mlp_counterfactual_flip_rate

assert mlp_cf_predictions == [0, 1, 1, 0]
assert cnn_cf_predictions == [0, 0, 1, 1]
assert selected_mlp_moon_counterfactual_flipped
assert selected_cnn_moon_counterfactual_stable
assert selected_mlp_star_counterfactual_flipped
assert selected_cnn_star_counterfactual_stable
assert selected_cnn_moon_moved_correct_prob >= 0.90
assert selected_cnn_star_moved_correct_prob >= 0.90
assert selected_mlp_moon_moved_wrong_prob >= 0.90
assert selected_mlp_star_moved_wrong_prob >= 0.90

assert "moon_position_response_mlp" in globals()
assert "star_position_response_mlp" in globals()
assert "position_response_metrics" in globals()
assert "decision_boundary_results" in globals()
assert "shape_position_surface_mlp" in globals()
assert "shape_position_surface_cnn" in globals()
assert "shape_morph_results" in globals()
assert "movement_path_results" in globals()
assert "attribution_density_results" in globals()
assert "average_relevance_results" in globals()
assert "representation_neighbour_results" in globals()
assert "representation_probe_results" in globals()
assert len(minimal_masking_results) > 0
assert best_patch_drop >= 0.0
assert best_masking_result["ranked_score"] <= original_wrong_score + 1e-8
assert mlp_object_mass > 0.05
assert mlp_position_sensitivity > cnn_position_sensitivity
assert cnn_shape_consistency > mlp_shape_consistency
assert shape_morph_results["lower_left"]["cnn"][0] <= 0.10
assert shape_morph_results["lower_left"]["cnn"][-1] >= 0.90
assert shape_morph_results["upper_right"]["cnn"][0] <= 0.10
assert shape_morph_results["upper_right"]["cnn"][-1] >= 0.90

for result_name in ["representation_neighbour_results", "representation_probe_results"]:
    assert len(globals()[result_name]) > 0, result_name
for model_name in ["MLP", "CNN"]:
    for metric_name in ["shape", "position"]:
        assert np.isfinite(representation_probe_results[model_name][metric_name])

for filename in [
    "fig00_apparent_shape_task.png",
    "fig00_model_cards.png",
    "fig01_both_models_appear_to_work.png",
    "fig01_shape_position_grid.png",
    "fig02_training_position_distribution.png",
    "fig05_same_shape_moved.png",
    "fig06_position_response_maps.png",
    "fig07_position_decision_boundary.png",
    "fig08_shape_morph_strip.png",
    "fig09_shape_position_surface.png",
    "fig10_movement_path.png",
    "fig11_saliency_comparison.png",
    "fig12_average_relevance_maps.png",
    "fig13_representation_neighbours.png",
    "fig14_representation_probes.png",
    "fig15_minimal_evidence_removal.png",
    "fig16_what_changes_the_decision.png",
    "fig17_evidence_ledger.png",
    "fig20_act2_apparent_shape_task_invisible_background.png",
    "fig20a_background_only_sanity_check.png",
    "fig21_act2_cnn_appears_to_work.png",
    "fig22_invisible_background_swap_counterfactual.png",
    "fig23_invisible_background_confidence_sweep.png",
    "fig24_invisible_background_difference_amplified.png",
    "fig25_background_tint_response_surface.png",
    "fig26_background_vs_shape_bars.png",
    "fig27_act2_heatmaps_are_not_enough.png",
    "fig28_act2_mitigation_retest.png",
    "fig29_two_act_evidence_board.png",
    "fig_final_xai_loop.png",
    "fig_final_real_world_bridge.png",
]:
    assert (FIGURE_DIR / filename).exists(), filename

assert "presentation_export_manifest" in globals()
assert "suggested_slide_order" in globals()
for filename in PRESENTATION_FILES:
    assert filename in presentation_export_manifest, filename
    assert (FIGURE_DIR / filename).exists(), filename
assert len(suggested_slide_order) == 20
assert "anim_moon_moves_confidence.mp4" in presentation_export_manifest
assert "anim_star_moves_confidence.mp4" in presentation_export_manifest
assert "anim_moon_moves_heatmaps.mp4" in presentation_export_manifest
assert "anim_star_moves_heatmaps.mp4" in presentation_export_manifest
assert "anim_morph_lower_left_heatmaps.mp4" in presentation_export_manifest
assert "anim_morph_upper_right_heatmaps.mp4" in presentation_export_manifest
assert "fig_final_bridge_to_real_xai.png" in presentation_export_manifest
assert "anim_invisible_background_morph_moon.mp4" in presentation_export_manifest
assert "anim_invisible_background_morph_star.mp4" in presentation_export_manifest
assert act2_validation_accuracy >= 0.995
assert act2_validation_mean_correct_prob >= 0.98
assert act2_validation_min_correct_prob >= 0.90
assert act2_mitigated_validation_accuracy >= 0.90
assert act2_biased_swap_gap > act2_mitigated_swap_gap
assert act2_swap_scores["moon_on_moon_bg"] <= 0.05
assert act2_swap_scores["moon_on_star_bg"] >= 0.95
assert act2_swap_scores["star_on_star_bg"] >= 0.95
assert act2_swap_scores["star_on_moon_bg"] <= 0.05
assert act2_mitigated_swap_scores["moon_on_moon_bg"] <= 0.10
assert act2_mitigated_swap_scores["moon_on_star_bg"] <= 0.10
assert act2_mitigated_swap_scores["star_on_star_bg"] >= 0.90
assert act2_mitigated_swap_scores["star_on_moon_bg"] >= 0.90
NOTEBOOK_TEXT = NOTEBOOK_PATH.read_text(encoding="utf-8")
assert "Same moon. Same pixels. Different place. Different answer." in NOTEBOOK_TEXT
assert "Same shape. Almost invisible background shift. Different belief." in NOTEBOOK_TEXT
assert "The shortcut changes. The XAI discipline does not." in NOTEBOOK_TEXT
assert "Accuracy tells us whether the model was right on the exam. XAI asks whether it learned the rule we meant to teach." in NOTEBOOK_TEXT
assert "Unmasking Clever Hans predictors" in NOTEBOOK_TEXT
assert "Sanity Checks for Saliency Maps" in NOTEBOOK_TEXT
FORBIDDEN_OLD_SHORTCUT_PHRASES = [
    "blue " + "scene",
    "amber " + "scene",
    "moon on " + "blue",
    "moon on " + "amber",
    "star on " + "blue",
    "star on " + "amber",
    "SCENE" + "_NAMES",
    "scene " + "cue",
]
for phrase in FORBIDDEN_OLD_SHORTCUT_PHRASES:
    assert phrase not in NOTEBOOK_TEXT, f"Forbidden old shortcut phrase remains: {phrase}"

print("Demo 00 self-check passed.")
